# Coleta — Pipeline CheckAI Autoral (Verdadeiros) v3

Coleta e estrutura candidatos a **claims verdadeiros** próprios do CheckAI a partir de fontes
oficiais e jornalísticas de acesso público.

**Objetivo:** expandir a base própria do CheckAI com `label=1` (verdadeiros) de alta qualidade,
rastreaáveis e verificáveis, para reduzir o gap V2_PROPRIA (f1=0.7569 na Fase 7).

**Melhorias v3 (em relação à v2):**
- `VERBOS_DECLARATORIOS`: conjunto explícito de verbos verificados por palavra exata (`\b`)
- Declarações têm **prioridade absoluta**: mesmo com verbo factual, DECLARACAO_PUBLICA vence
- `VERBOS_FACTUAIS_APROVADO_AUTO`: conjunto restrito — apenas ações institucionais objetivas
- `PADROES_PAGINA`: páginas estáticas, hotsites e relatórios sem contexto → DESCARTADO
- `PADROES_PENDENTE` expandido: conheça, tire dúvidas, perguntas e respostas, etc.
- `texto_principal_modelo` para DECLARACAO_PUBLICA: normaliza verbo declaratório para passado
- Nova coluna `tipo_claim`: FATO_INSTITUCIONAL / DECLARACAO_PUBLICA / CHAMADA_EXPLICATIVA / PAGINA_ESTATICA / OUTRO_PENDENTE
- Relatório com seção de impacto v2→v3

**Regras obrigatórias:**
- Não criar V4. Não treinar modelo. Não alterar V1/V2/V3.
- Toda linha precisa de `url_origem` preenchida.
- Não usar IA como fonte de verdade.
- Pipeline determinístico, baseado em regras.
- Manter arquivos antigos preservados.

## Bibliotecas

In [25]:
import re
import uuid
import time
import xml.etree.ElementTree as ET
import requests
import feedparser
import pandas as pd

from datetime import datetime
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urlparse, urljoin
from email.utils import parsedate_to_datetime

## Configuração

In [26]:
SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "src" else _cwd

NOME_PIPELINE = "pipeline_checkai_autoral"
PASTA_RAW     = PROJECT_ROOT / "dados" / NOME_PIPELINE / "raw"
PASTA_CURATED = PROJECT_ROOT / "dados" / NOME_PIPELINE / "curated"
PASTA_FINAL   = PROJECT_ROOT / "dados" / NOME_PIPELINE / "final"

for pasta in [PASTA_RAW, PASTA_CURATED, PASTA_FINAL]:
    pasta.mkdir(parents=True, exist_ok=True)

USER_AGENT      = "CheckAI-TCC-Research/1.0 (academic; UNICID; non-commercial)"
REQUEST_TIMEOUT = 10

# ─── Parâmetros de volume (RSS) ───────────────────────────────────────────
MAX_ITENS_POR_FONTE = 200   # máximo de itens por feed RSS
MAX_ITENS_TOTAL     = 1000  # limite global de registros brutos
DELAY_SEGUNDOS      = 1.5   # delay entre requisições (segundos)
# ──────────────────────────────────────────────────────────────────────────

# ─── Parâmetros de coleta histórica ──────────────────────────────────────
MAX_PAGINAS_POR_FONTE = 10          # máximo de páginas/sitemaps por fonte histórica
DATA_MIN_HISTORICO    = "2025-01-01"  # só coletar notícias a partir desta data
# ──────────────────────────────────────────────────────────────────────────

DELAY_ENTRE_REQUESTS = DELAY_SEGUNDOS  # alias interno (mantido por compatibilidade)

print(f"TIMESTAMP            : {TIMESTAMP}")
print(f"PROJECT_ROOT         : {PROJECT_ROOT}")
print(f"MAX_ITENS_POR_FONTE  : {MAX_ITENS_POR_FONTE}")
print(f"MAX_ITENS_TOTAL      : {MAX_ITENS_TOTAL}")
print(f"DELAY_SEGUNDOS       : {DELAY_SEGUNDOS}s")
print(f"MAX_PAGINAS_POR_FONTE: {MAX_PAGINAS_POR_FONTE}")
print(f"DATA_MIN_HISTORICO   : {DATA_MIN_HISTORICO}")

TIMESTAMP            : 2026-05-30_21-04-39
PROJECT_ROOT         : C:\Users\offan\Desktop\ml-checkai
MAX_ITENS_POR_FONTE  : 200
MAX_ITENS_TOTAL      : 1000
DELAY_SEGUNDOS       : 1.5s
MAX_PAGINAS_POR_FONTE: 10
DATA_MIN_HISTORICO   : 2025-01-01


## Constantes — padrões de classificação e verbos

### Hierarquia de classificação v3 (ordem de aplicação em `gerar_claim`):

```
1. Vazio/muito curto            → DESCARTADO
2. PADROES_DESCARTADO           → DESCARTADO   (opinião, análise, editorial)
2b. PADROES_PAGINA              → DESCARTADO   (página estática, hotsite, relatório)
3. PADROES_DECLARACAO (***)     → PENDENTE_REVISAO + DECLARACAO_PUBLICA
4. PADROES_PENDENTE             → PENDENTE_REVISAO + VERDADEIRO_CURADO
5. Sem verbo factual            → PENDENTE_REVISAO
6. Verbo factual identificado   → APROVADO_AUTO + VERDADEIRO_CURADO
```

(***) Declaração tem prioridade absoluta: mesmo que haja verbo factual, se houver verbo
declaratório, o título vai para DECLARACAO_PUBLICA.

In [27]:
TEMAS_PRIORITARIOS = [
    # 1. Eleições e Justiça Eleitoral
    "eleições 2026", "urna eletrônica", "fraude eleitoral", "TSE",
    "biometria eleitoral", "título de eleitor", "propaganda eleitoral",
    "pesquisa eleitoral", "voto impresso", "cassação eleitoral",
    "inelegibilidade", "fake news eleitoral", "deepfake eleições",
    "inteligência artificial eleições",
    # 2. Congresso, governo e instituições
    "Congresso Nacional", "Câmara dos Deputados", "Senado Federal",
    "STF", "Alexandre de Moraes", "Supremo Tribunal Federal",
    "Polícia Federal", "8 de janeiro", "impeachment", "CPI", "PEC",
    "projeto de lei", "medida provisória", "emendas parlamentares",
    "orçamento secreto", "reforma política",
    # 3. Pautas populares e trabalhistas
    "escala 6x1", "fim da escala 6x1", "jornada de trabalho", "CLT",
    "salário mínimo", "imposto de renda", "INSS", "aposentadoria", "FGTS",
    "seguro-desemprego", "MEI", "pejotização", "direitos trabalhistas",
    "reforma trabalhista",
    # 4. Economia popular
    "Pix", "Banco Central", "inflação", "taxa Selic", "dólar",
    "preço da gasolina", "preço dos alimentos", "cesta básica",
    "Bolsa Família", "Auxílio Brasil", "Minha Casa Minha Vida",
    "Desenrola Brasil", "reforma tributária",
    "imposto sobre compras internacionais", "taxação da Shein",
    "taxação de bets", "apostas online", "dívida pública",
    # 5. Saúde, educação e segurança pública
    "SUS", "vacina", "vacinação", "dengue", "Covid", "saúde pública",
    "educação pública", "Enem", "Pé-de-Meia", "Fies", "Prouni",
    "segurança pública", "saidinha temporária", "porte de armas",
    "drogas", "aborto",
    # 6. Figuras políticas
    "Lula", "Bolsonaro", "Jair Bolsonaro", "Michelle Bolsonaro",
    "Eduardo Bolsonaro", "Flávio Bolsonaro", "Carlos Bolsonaro",
    "Tarcísio de Freitas", "Guilherme Boulos", "Pablo Marçal",
    "Nikolas Ferreira", "André Janones", "Arthur Lira",
    "Rodrigo Pacheco", "Hugo Motta", "Alexandre de Moraes", "Dino",
    "Sergio Moro", "Damares Alves", "Marina Silva",
    # 7. Movimentos, partidos e grupos políticos
    "MBL", "Movimento Brasil Livre", "PT", "PL", "PSOL", "MDB",
    "União Brasil", "Republicanos", "Novo", "Centrão", "esquerda",
    "direita", "extrema direita", "extrema esquerda", "comunismo",
    "socialismo", "conservadores", "bolsonarismo", "lulismo",
    "lava jato",
    # 8. Temas virais e narrativas de redes sociais
    "mamata", "rachadinha", "gabinete do ódio", "kit gay",
    "ideologia de gênero", "comunismo no Brasil", "fraude nas urnas",
    "Venezuela", "Foro de São Paulo", "censura",
    "liberdade de expressão", "bloqueio de redes sociais", "fake news",
    "desinformação", "vídeo manipulado", "áudio falso", "print falso",
    "montagem política",
]

# ── QUERIES_SUGERIDAS — lista estruturada com tema e subtema ─────────────
QUERIES_SUGERIDAS = [
    # ─── Pautas trabalhistas ─────────────────────────────────────────────
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "escala 6x1"},
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "fim da escala 6x1"},
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "jornada de trabalho"},
    {"tema": "trabalhista",         "subtema": "legislacao",         "query": "CLT"},
    {"tema": "trabalhista",         "subtema": "remuneracao",        "query": "salário mínimo"},
    {"tema": "trabalhista",         "subtema": "previdencia",        "query": "INSS"},
    {"tema": "trabalhista",         "subtema": "previdencia",        "query": "aposentadoria"},
    {"tema": "trabalhista",         "subtema": "beneficios",         "query": "FGTS"},
    {"tema": "trabalhista",         "subtema": "empreendedorismo",   "query": "MEI"},
    {"tema": "trabalhista",         "subtema": "legislacao",         "query": "pejotização"},
    # ─── Eleições e Justiça Eleitoral ────────────────────────────────────
    {"tema": "eleitoral",           "subtema": "eleicoes_2026",      "query": "eleições 2026"},
    {"tema": "eleitoral",           "subtema": "sistema_voto",       "query": "urna eletrônica"},
    {"tema": "eleitoral",           "subtema": "fraude_eleitoral",   "query": "fraude eleitoral"},
    {"tema": "eleitoral",           "subtema": "justica_eleitoral",  "query": "TSE"},
    {"tema": "eleitoral",           "subtema": "sistema_voto",       "query": "voto impresso"},
    {"tema": "eleitoral",           "subtema": "cadastro_eleitoral", "query": "biometria eleitoral"},
    {"tema": "eleitoral",           "subtema": "campanha",           "query": "propaganda eleitoral"},
    {"tema": "eleitoral",           "subtema": "pesquisas",          "query": "pesquisa eleitoral"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "fake news eleitoral"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "deepfake eleições"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "inteligência artificial eleições"},
    # ─── Congresso, governo e instituições ───────────────────────────────
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "STF"},
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "Alexandre de Moraes"},
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "Supremo Tribunal Federal"},
    {"tema": "institucional",       "subtema": "politica_criminal",  "query": "8 de janeiro"},
    {"tema": "institucional",       "subtema": "policia",            "query": "Polícia Federal"},
    {"tema": "institucional",       "subtema": "poder_executivo",    "query": "impeachment"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "CPI"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "PEC"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Congresso Nacional"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Câmara dos Deputados"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Senado Federal"},
    {"tema": "institucional",       "subtema": "orcamento",          "query": "emendas parlamentares"},
    {"tema": "institucional",       "subtema": "orcamento",          "query": "orçamento secreto"},
    # ─── Economia popular ─────────────────────────────────────────────────
    {"tema": "economia_popular",    "subtema": "pagamentos_digitais","query": "Pix"},
    {"tema": "economia_popular",    "subtema": "politica_monetaria", "query": "Banco Central"},
    {"tema": "economia_popular",    "subtema": "indices",            "query": "inflação"},
    {"tema": "economia_popular",    "subtema": "politica_monetaria", "query": "taxa Selic"},
    {"tema": "economia_popular",    "subtema": "combustiveis",       "query": "preço da gasolina"},
    {"tema": "economia_popular",    "subtema": "assistencia_social", "query": "Bolsa Família"},
    {"tema": "economia_popular",    "subtema": "assistencia_social", "query": "Auxílio Brasil"},
    {"tema": "economia_popular",    "subtema": "habitacao",          "query": "Minha Casa Minha Vida"},
    {"tema": "economia_popular",    "subtema": "tributacao",         "query": "reforma tributária"},
    {"tema": "economia_popular",    "subtema": "tributacao",         "query": "imposto de renda"},
    {"tema": "economia_popular",    "subtema": "comercio_exterior",  "query": "taxação da Shein"},
    {"tema": "economia_popular",    "subtema": "regulacao",          "query": "taxação de bets"},
    # ─── Saúde, educação e segurança pública ──────────────────────────────
    {"tema": "social",              "subtema": "saude",              "query": "SUS"},
    {"tema": "social",              "subtema": "saude",              "query": "vacina"},
    {"tema": "social",              "subtema": "saude",              "query": "dengue"},
    {"tema": "social",              "subtema": "saude",              "query": "Covid"},
    {"tema": "social",              "subtema": "educacao",           "query": "Enem"},
    {"tema": "social",              "subtema": "educacao",           "query": "Fies"},
    {"tema": "social",              "subtema": "educacao",           "query": "Prouni"},
    {"tema": "social",              "subtema": "seguranca",          "query": "segurança pública"},
    {"tema": "social",              "subtema": "seguranca",          "query": "saidinha temporária"},
    {"tema": "social",              "subtema": "seguranca",          "query": "porte de armas"},
    # ─── Figuras políticas ────────────────────────────────────────────────
    {"tema": "politica_figuras",    "subtema": "governo_federal",    "query": "Lula"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Jair Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Michelle Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Eduardo Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "governos_estaduais", "query": "Tarcísio de Freitas"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Guilherme Boulos"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Pablo Marçal"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Nikolas Ferreira"},
    {"tema": "politica_figuras",    "subtema": "governo_federal",    "query": "André Janones"},
    {"tema": "politica_figuras",    "subtema": "legislativo",        "query": "Arthur Lira"},
    {"tema": "politica_figuras",    "subtema": "legislativo",        "query": "Rodrigo Pacheco"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Sergio Moro"},
    # ─── Movimentos, partidos e grupos políticos ──────────────────────────
    {"tema": "politica_partidos",   "subtema": "movimentos",         "query": "MBL"},
    {"tema": "politica_partidos",   "subtema": "movimentos",         "query": "Movimento Brasil Livre"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PT"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PL"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PSOL"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "MDB"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "Novo"},
    {"tema": "politica_partidos",   "subtema": "legislativo",        "query": "Centrão"},
    {"tema": "politica_partidos",   "subtema": "ideologia",          "query": "bolsonarismo"},
    {"tema": "politica_partidos",   "subtema": "ideologia",          "query": "lulismo"},
    {"tema": "politica_partidos",   "subtema": "investigacoes",      "query": "lava jato"},
    # ─── Temas virais e narrativas de redes sociais ───────────────────────
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "kit gay"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "ideologia de gênero"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "comunismo no Brasil"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "Foro de São Paulo"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "Venezuela"},
    {"tema": "desinformacao_viral", "subtema": "liberdade_expressao","query": "censura"},
    {"tema": "desinformacao_viral", "subtema": "liberdade_expressao","query": "liberdade de expressão"},
    {"tema": "desinformacao_viral", "subtema": "regulacao_digital",  "query": "bloqueio de redes sociais"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "vídeo manipulado"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "áudio falso"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "print falso"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "montagem política"},
]


# ── PADROES_DESCARTADO ────────────────────────────────────────
# Opinião, análise ou editorial → DESCARTADO (motivo: opiniao_ou_analise)
PADROES_DESCARTADO = [
    r"^opini[aã]o\b",
    r"^coluna\b",
    r"^editorial\b",
    r"^an[aá]lise\b",
    r"^reflex[aã]o\b",
    r"^coment[aá]rio\b",
    r"^artigo\b",
    r"^tribuna\b",
    r"^ponto\s+de\s+vista\b",
    r"^debate\b",
]

# ── PADROES_PAGINA ───────────────────────────────────────────
# Páginas estáticas, hotsites, relatórios sem contexto → DESCARTADO
PADROES_PAGINA = [
    r"^hotsite\b",
    r"\bhotsite\b",
    r"^p[aá]gina\s+(do|da|de|dos|das)\b",
    r"^uso\s+(do|da)\b",
    r"^ber[cç][aá]rio\b",
    r"^museu\b",
    r"^relat[oó]rio\s*[–—\-]\s*\d",
]

# ── VERBOS_DECLARATORIOS ──────────────────────────────────────
# Prioridade absoluta: se presente, vai para DECLARACAO_PUBLICA
VERBOS_DECLARATORIOS = {
    "diz", "disse",
    "afirma", "afirmou",
    "declara", "declarou",
    "defende", "defendeu",
    "critica", "criticou",
    "cobra", "cobrou",
    "acusa", "acusou",
    "nega", "negou",
    "avalia", "avaliou",
    "promete", "prometeu",
    "garante", "garantiu",
    "alerta", "alertou",
    "adverte", "advertiu",
    "prevê", "previu",
    "aponta", "apontou",
    "ressalta", "ressaltou",
    "destaca", "destacou",
    "sugere", "sugeriu",
    "indica", "indicou",
    "pede", "pediu",
    "exige", "exigiu",
    "reclama", "reclamou",
    "lamenta", "lamentou",
    "rebate", "rebateu",
    "contesta", "contestou",
    "reconhece", "reconheceu",
    "admite", "admitiu",
}

FRASES_DECLARATORIAS = [
    r'^"',
    "^\u201c",
    "^\u2018",
    r"\bsegundo\s+\w",
    r"na\s+avalia[cç][aã]o\s+de",
    r"\bpara\s+(o\s+|a\s+)?ministro\b",
    r"\bpara\s+(o\s+|a\s+)?presidente\b",
    r"\bpara\s+(o\s+|a\s+)?especialista\b",
    r"\bpara\s+(o\s+|a\s+)?senador\b",
    r"\bpara\s+(o\s+|a\s+)?deputado\b",
    r"\bpara\s+(o\s+|a\s+)?governador\b",
    r"\bpara\s+(o\s+|a\s+)?economista\b",
    r"\bpara\s+(o\s+|a\s+)?analista\b",
]

VERBOS_DECLARATORIOS_PASSADO = {
    "diz": "disse",        "afirma": "afirmou",
    "declara": "declarou", "defende": "defendeu",
    "critica": "criticou", "cobra": "cobrou",
    "acusa": "acusou",     "nega": "negou",
    "avalia": "avaliou",   "promete": "prometeu",
    "garante": "garantiu", "alerta": "alertou",
    "adverte": "advertiu", "prevê": "previu",
    "aponta": "apontou",   "ressalta": "ressaltou",
    "destaca": "destacou", "sugere": "sugeriu",
    "indica": "indicou",   "pede": "pediu",
    "exige": "exigiu",     "reclama": "reclamou",
    "lamenta": "lamentou", "rebate": "rebateu",
    "contesta": "contestou", "reconhece": "reconheceu",
    "admite": "admitiu",
}

# ── PADROES_PENDENTE ──────────────────────────────────────────
PADROES_PENDENTE = [
    r"^entenda\b", r"^veja\b", r"^saiba\b", r"^confira\b",
    r"^conhe[cç]a\b", r"^tire\s+d[uú]vidas\b",
    r"^como\b", r"^por\s+que\b", r"^o\s+que\b",
    r"^quem\b", r"^quando\b", r"^onde\b",
    r"^guia\b",
    r"\?",
    r"^ao\s+vivo\b", r"^podcast\b", r"^v[ií]deo\b",
    r"^\d+\s+(coisas|fatos|razões|dicas)\b",
    r"\bperguntas\s+e\s+respostas\b",
    r"\bo\s+que\s+muda\b",
    r"\bcomo\s+funciona\b",
    r"\bo\s+que\s+[eé]\b",
    r"^especial[\s:–—\-]",
]

VERBOS_PRESENTE_PASSADO = {
    "aprova": "aprovou",       "veta": "vetou",
    "divulga": "divulgou",     "anuncia": "anunciou",
    "mantém": "manteve",       "apresenta": "apresentou",
    "suspende": "suspendeu",   "decide": "decidiu",
    "confirma": "confirmou",   "rejeita": "rejeitou",
    "vota": "votou",           "publica": "publicou",
    "cria": "criou",           "lança": "lançou",
    "determina": "determinou", "proíbe": "proibiu",
    "autoriza": "autorizou",   "sanciona": "sancionou",
    "promulga": "promulgou",   "regulamenta": "regulamentou",
    "assina": "assinou",       "amplia": "ampliou",
    "reduz": "reduziu",        "aumenta": "aumentou",
    "cancela": "cancelou",     "inicia": "iniciou",
    "encerra": "encerrou",     "libera": "liberou",
    "fixa": "fixou",           "institui": "instituiu",
    "retoma": "retomou",       "revoga": "revogou",
    "estabelece": "estabeleceu", "nomeia": "nomeou",
    "exonera": "exonerou",     "abre": "abriu",
    "elege": "elegeu",         "propõe": "propôs",
    "recomenda": "recomendou", "atinge": "atingiu",
    "alcança": "alcançou",     "sobe": "subiu",
    "cai": "caiu",             "paga": "pagou",
    "emite": "emitiu",         "edita": "editou",
    "fecha": "fechou",         "derruba": "derrubou",
    "prorroga": "prorrogou",   "extingue": "extinguiu",
    "convoca": "convocou",     "instala": "instalou",
    "destitui": "destituiu",   "recua": "recuou",
    "homologa": "homologou",   "valida": "validou",
    "implementa": "implementou", "digitaliza": "digitalizou",
    "moderniza": "modernizou", "registra": "registrou",
    "atualiza": "atualizou",   "eleva": "elevou",
    "corta": "cortou",         "investe": "investiu",
    "destina": "destinou",     "transfere": "transferiu",
    "reestrutura": "reestruturou",
}

VERBOS_FACTUAIS_APROVADO_AUTO = {
    "aprova", "sanciona", "publica", "divulga", "lança", "cria",
    "institui", "regulamenta", "decide", "mantém", "rejeita", "autoriza",
    "suspende", "prorroga", "retoma", "amplia", "reduz", "registra",
    "confirma", "vota", "promulga", "assina", "determina", "proíbe",
    "libera", "fixa", "revoga", "estabelece", "nomeia", "exonera",
    "elege", "convoca", "instala", "cancela", "encerra", "abre",
    "homologa", "valida", "implementa", "digitaliza", "moderniza",
    "atualiza", "eleva", "corta", "investe", "destina", "transfere",
    "veta", "anuncia",
    "aprovou", "sancionou", "publicou", "divulgou", "lançou", "criou",
    "instituiu", "regulamentou", "decidiu", "manteve", "rejeitou", "autorizou",
    "suspendeu", "prorrogou", "retomou", "ampliou", "reduziu", "registrou",
    "confirmou", "votou", "promulgou", "assinou", "determinou", "proibiu",
    "liberou", "fixou", "revogou", "estabeleceu", "nomeou", "exonerou",
    "elegeu", "convocou", "instalou", "cancelou", "encerrou", "abriu",
    "homologou", "validou", "implementou", "digitalizou", "modernizou",
    "atualizou", "elevou", "cortou", "investiu", "destinou", "transferiu",
    "vetou", "anunciou",
}

FONTES_FORTES = {
    "TSE", "STF", "CAMARA_NOTICIAS", "AGENCIA_CAMARA",
    "SENADO_NOTICIAS", "AGENCIA_BRASIL", "BANCO_CENTRAL", "IBGE",
    "GOV_FAZENDA", "GOV_SAUDE", "GOV_EDUCACAO", "GOV_PREVIDENCIA", "CNJ",
}

CLAIM_MIN_CHARS = 40

print(f"Temas prioritários           : {len(TEMAS_PRIORITARIOS)}")
print(f"Queries sugeridas            : {len(QUERIES_SUGERIDAS)}")
print(f"Temas únicos (queries)       : {len({q['tema'] for q in QUERIES_SUGERIDAS})}")
print(f"Subtemas únicos (queries)    : {len({q['subtema'] for q in QUERIES_SUGERIDAS})}")
print(f"Padrões descartado (opinião) : {len(PADROES_DESCARTADO)}")
print(f"Padrões descartado (página) : {len(PADROES_PAGINA)}")
print(f"Verbos declaratórios         : {len(VERBOS_DECLARATORIOS)}")
print(f"Frases declaratórias         : {len(FRASES_DECLARATORIAS)}")
print(f"Padrões pendente             : {len(PADROES_PENDENTE)}")
print(f"Verbos factuais (aprovação) : {len(VERBOS_FACTUAIS_APROVADO_AUTO)}")
print(f"Fontes fortes                : {len(FONTES_FORTES)}")

Temas prioritários           : 136
Queries sugeridas            : 92
Temas únicos (queries)       : 8
Subtemas únicos (queries)    : 43
Padrões descartado (opinião) : 10
Padrões descartado (página) : 7
Verbos declaratórios         : 54
Frases declaratórias         : 13
Padrões pendente             : 23
Verbos factuais (aprovação) : 98
Fontes fortes                : 13


## Configuração das fontes

| Portal | Tipo | Método | origem_qualidade |
|---|---|---|---|
| Agência Brasil (política, economia, saúde, educação, direitos, geral, ciência) | Jornalismo público EBC | RSS | ROTULO_ASSUMIDO_ALTO |
| STF Notícias | Oficial — Judiciário | RSS | ROTULO_ASSUMIDO_ALTO |
| Câmara Notícias | Oficial — Legislativo | RSS | ROTULO_ASSUMIDO_ALTO |
| Senado Notícias | Oficial — Legislativo | RSS | ROTULO_ASSUMIDO_ALTO |
| TSE | Oficial — Justiça Eleitoral | RSS | ROTULO_ASSUMIDO_ALTO |
| IBGE Agência | Oficial — Estatísticas | RSS | ROTULO_ASSUMIDO_ALTO |
| Banco Central | Oficial — Política monetária | API JSON | ROTULO_ASSUMIDO_ALTO |

In [28]:
FONTES_RSS = [
    # ─── Agência Brasil — política ─────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "política", "subtema": "governo_federal",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/politica/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — economia ─────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "economia", "subtema": "economia_nacional",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/economia/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — saúde ────────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "saúde_pública", "subtema": "saude_publica",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/saude/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — educação ─────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "educação", "subtema": "educacao",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/educacao/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — direitos humanos ─────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "direitos_humanos", "subtema": "direitos_e_justica",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/direitos-humanos/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — justiça ──────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "judiciário", "subtema": "justica",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/justica/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — geral ────────────────────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "geral", "subtema": "geral",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/geral/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — ciência e tecnologia ─────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "tecnologia", "subtema": "ciencia_tecnologia",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/ciencia-e-tecnologia/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Agência Brasil — pesquisa e inovação ──────────────────────────────
    {
        "portal": "AGENCIA_BRASIL", "tema": "tecnologia", "subtema": "pesquisa_inovacao",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/pesquisa-e-inovacao/feed.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── STF Notícias ──────────────────────────────────────────────────────
    {
        "portal": "STF", "tema": "judiciário", "subtema": "decisoes_stf",
        "url_feed": "https://noticias.stf.jus.br/feed/",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Câmara Notícias (URL principal com barra) ─────────────────────────
    {
        "portal": "CAMARA_NOTICIAS", "tema": "legislativo", "subtema": "camara_dos_deputados",
        "url_feed": "https://www.camara.leg.br/noticias/rss/",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Câmara Notícias (URL alternativa — fallback) ──────────────────────
    {
        "portal": "CAMARA_NOTICIAS", "tema": "legislativo", "subtema": "camara_dos_deputados",
        "url_feed": "https://www.camara.leg.br/noticias/rss",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
        "_fallback": True,
    },
    # ─── Agência Câmara (serviço jornalístico distinto da Câmara Notícias) ─
    {
        "portal": "AGENCIA_CAMARA", "tema": "legislativo", "subtema": "agencia_camara",
        "url_feed": "https://agencia.camara.leg.br/feed/",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Senado Notícias ───────────────────────────────────────────────────
    {
        "portal": "SENADO_NOTICIAS", "tema": "legislativo", "subtema": "senado_federal",
        "url_feed": "https://www12.senado.leg.br/noticias/rss.xml",
        "preferencia_campo": "title_only", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── TSE Notícias ──────────────────────────────────────────────────────
    {
        "portal": "TSE", "tema": "eleitoral", "subtema": "eleicoes_2026",
        "url_feed": "https://www.tse.jus.br/comunicacao/noticias/rss",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── IBGE Agência de Notícias ──────────────────────────────────────────
    {
        "portal": "IBGE", "tema": "economia", "subtema": "indicadores_oficiais",
        "url_feed": "https://agencia.ibge.gov.br/rssFeed.xml",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Gov.br — Ministério da Fazenda ────────────────────────────────────
    {
        "portal": "GOV_FAZENDA", "tema": "economia", "subtema": "ministerio_fazenda",
        "url_feed": "https://www.gov.br/fazenda/pt-br/assuntos/noticias/feed.xml",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Gov.br — Ministério da Saúde ──────────────────────────────────────
    {
        "portal": "GOV_SAUDE", "tema": "saúde_pública", "subtema": "ministerio_saude",
        "url_feed": "https://www.gov.br/saude/pt-br/assuntos/noticias/feed.xml",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Gov.br — Ministério da Educação (MEC) ─────────────────────────────
    {
        "portal": "GOV_EDUCACAO", "tema": "educação", "subtema": "ministerio_educacao",
        "url_feed": "https://www.gov.br/mec/pt-br/assuntos/noticias/feed.xml",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── Gov.br — Previdência Social ───────────────────────────────────────
    {
        "portal": "GOV_PREVIDENCIA", "tema": "previdência", "subtema": "ministerio_previdencia",
        "url_feed": "https://www.gov.br/previdencia/pt-br/assuntos/noticias/feed.xml",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    # ─── CNJ — Conselho Nacional de Justiça ────────────────────────────────
    {
        "portal": "CNJ", "tema": "judiciário", "subtema": "conselho_nacional_justica",
        "url_feed": "https://www.cnj.jus.br/feed/",
        "preferencia_campo": "summary", "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
]

fontes_principais = [f for f in FONTES_RSS if not f.get("_fallback", False)]
print(f"Fontes RSS configuradas: {len(fontes_principais)} principais + 1 fallback Câmara")
print()
print(f"{'Portal':<20} {'Tema/Subtema'}")
print("-" * 60)
for f in fontes_principais:
    print(f"  {f['portal']:<18} {f['tema']}/{f['subtema']}")

Fontes RSS configuradas: 20 principais + 1 fallback Câmara

Portal               Tema/Subtema
------------------------------------------------------------
  AGENCIA_BRASIL     política/governo_federal
  AGENCIA_BRASIL     economia/economia_nacional
  AGENCIA_BRASIL     saúde_pública/saude_publica
  AGENCIA_BRASIL     educação/educacao
  AGENCIA_BRASIL     direitos_humanos/direitos_e_justica
  AGENCIA_BRASIL     judiciário/justica
  AGENCIA_BRASIL     geral/geral
  AGENCIA_BRASIL     tecnologia/ciencia_tecnologia
  AGENCIA_BRASIL     tecnologia/pesquisa_inovacao
  STF                judiciário/decisoes_stf
  CAMARA_NOTICIAS    legislativo/camara_dos_deputados
  AGENCIA_CAMARA     legislativo/agencia_camara
  SENADO_NOTICIAS    legislativo/senado_federal
  TSE                eleitoral/eleicoes_2026
  IBGE               economia/indicadores_oficiais
  GOV_FAZENDA        economia/ministerio_fazenda
  GOV_SAUDE          saúde_pública/ministerio_saude
  GOV_EDUCACAO       educação/ministerio

## Funções utilitárias

In [29]:
class _StripHTML(HTMLParser):
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    return re.sub(r"\s+", " ", parser.get_text()).strip()


def padronizar_data(valor) -> str:
    """Converte data RFC 2822 ou ISO para YYYY-MM-DD."""
    if not isinstance(valor, str) or not valor.strip():
        return ""
    try:
        return parsedate_to_datetime(valor.strip()).strftime("%Y-%m-%d")
    except Exception:
        pass
    for fmt in ["%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d", "%d/%m/%Y"]:
        try:
            return datetime.strptime(valor.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


def normalizar_pontuacao(texto: str) -> str:
    """Normaliza espaços, aspas e garante ponto final."""
    texto = re.sub(r"\s+", " ", texto).strip()
    texto = texto.replace("\u2018", "'").replace("\u2019", "'")
    texto = texto.replace("\u201c", '"').replace("\u201d", '"')
    if texto and texto[-1] not in ".!":
        texto = texto + "."
    return texto


def calcular_faixa_tamanho(tamanho: int) -> str:
    if tamanho < 500:
        return "curto"
    elif tamanho < 2000:
        return "medio"
    elif tamanho < 5000:
        return "longo"
    else:
        return "muito_longo"


print("Funções utilitárias definidas.")

Funções utilitárias definidas.


## Função `gerar_claim` (v3)

**Regras de classificação v3 (aplicadas nesta ordem):**

| Condição | status_curadoria | label_detalhe | motivo_status | tipo_claim |
|---|---|---|---|---|
| Vazio ou < 10 chars | DESCARTADO | DESCARTADO | titulo_vazio_ou_muito_curto | — |
| Opinião / análise / editorial | DESCARTADO | DESCARTADO | opiniao_ou_analise | — |
| Página estática / hotsite | DESCARTADO | DESCARTADO | pagina_ou_servico_institucional | PAGINA_ESTATICA |
| **Verbo declaratório** (prioridade absoluta) | PENDENTE_REVISAO | DECLARACAO_PUBLICA | declaracao_publica_requer_verificacao | DECLARACAO_PUBLICA |
| Interrogativo / entenda / conheça | PENDENTE_REVISAO | VERDADEIRO_CURADO | padrao_nao_factual_detectado | CHAMADA_EXPLICATIVA |
| Sem verbo factual (fonte não forte) | PENDENTE_REVISAO | VERDADEIRO_CURADO | sem_verbo_factual_fonte_nao_prioritaria | OUTRO_PENDENTE |
| Título curto sem verbo (fonte forte) | PENDENTE_REVISAO | VERDADEIRO_CURADO | titulo_muito_curto_sem_verbo | OUTRO_PENDENTE |
| Claim < 40 chars após transformação | PENDENTE_REVISAO | VERDADEIRO_CURADO | claim_muito_curto | OUTRO_PENDENTE |
| Verbo factual institucional | APROVADO_AUTO | VERDADEIRO_CURADO | claim_factual_gerado | FATO_INSTITUCIONAL |

**Bugs corrigidos em relação à v2:**
- `avalia`, `avaliou`, `garante`, `garantiu` etc. não são mais verbos factuais (são declaratórios)
- Verificação de declaração ocorre antes de qualquer check de verbo factual
- `texto_principal_modelo` para DECLARACAO_PUBLICA normaliza para passado (diz → disse)

In [30]:
def _tem_padrao(titulo_lower: str, padroes: list) -> bool:
    return any(re.search(p, titulo_lower) for p in padroes)


def _tem_padrao_declaratorio(titulo_lower: str) -> bool:
    """Verifica verbo declaratório (palavra exata) ou frase declaratória."""
    palavras = re.split(r"[\s,;:()—–\[\]]+", titulo_lower)
    if any(p in VERBOS_DECLARATORIOS for p in palavras if p):
        return True
    return any(re.search(p, titulo_lower) for p in FRASES_DECLARATORIAS)


def _tem_verbo_factual(titulo_lower: str) -> bool:
    """Verifica verbo factual institucional (conjunto estrito v3)."""
    palavras = re.split(r"[\s,;:()—–\[\]]+", titulo_lower)
    return any(p in VERBOS_FACTUAIS_APROVADO_AUTO for p in palavras if p)


def _converter_verbo_passado(titulo: str) -> str:
    """Converte verbo presente → passado para APROVADO_AUTO."""
    for presente, passado in VERBOS_PRESENTE_PASSADO.items():
        padrao = r"\b" + re.escape(presente) + r"\b"
        if re.search(padrao, titulo, flags=re.IGNORECASE):
            def _sub(m, p=passado):
                return p.capitalize() if m.group(0)[0].isupper() else p
            return re.sub(padrao, _sub, titulo, count=1, flags=re.IGNORECASE)
    return titulo


def _converter_declaratorio_passado(titulo: str) -> str:
    """Converte verbo declaratório presente → passado (para texto_principal_modelo)."""
    for presente, passado in VERBOS_DECLARATORIOS_PASSADO.items():
        padrao = r"\b" + re.escape(presente) + r"\b"
        if re.search(padrao, titulo, flags=re.IGNORECASE):
            def _sub(m, p=passado):
                return p.capitalize() if m.group(0)[0].isupper() else p
            return re.sub(padrao, _sub, titulo, count=1, flags=re.IGNORECASE)
    return titulo


def gerar_claim(titulo: str, resumo: str = "", portal: str = "") -> dict:
    """
    Transforma título em afirmação curta e factual (v3).
    Regras mais rígidas: declarações públicas têm prioridade absoluta.
    Não inventa fatos. Não usa IA como fonte de verdade.
    """
    titulo = remover_html(titulo).strip()
    resumo = remover_html(resumo).strip()

    def _ret(texto, status, label_det, motivo, origem):
        return {
            "texto_principal_modelo": texto,
            "status_curadoria":       status,
            "label_detalhe":          label_det,
            "motivo_status":          motivo,
            "origem_texto":           origem,
        }

    # 1. Descartar: vazio ou muito curto
    if not titulo or len(titulo) < 10:
        return _ret("", "DESCARTADO", "DESCARTADO", "titulo_vazio_ou_muito_curto", "")

    titulo_lower = titulo.lower()
    fonte_forte  = portal in FONTES_FORTES

    # 2. Descartar: opinião / análise / editorial
    if _tem_padrao(titulo_lower, PADROES_DESCARTADO):
        return _ret(titulo, "DESCARTADO", "DESCARTADO", "opiniao_ou_analise", "titulo")

    # 2b. Descartar: página estática / hotsite / relatório sem contexto
    if _tem_padrao(titulo_lower, PADROES_PAGINA):
        return _ret(titulo, "DESCARTADO", "DESCARTADO", "pagina_ou_servico_institucional", "titulo")

    # 3. Pendente: declaração pública (PRIORIDADE ABSOLUTA — antes de qualquer verbo factual)
    if _tem_padrao_declaratorio(titulo_lower):
        claim_decl = _converter_declaratorio_passado(titulo)
        claim_decl = normalizar_pontuacao(claim_decl)
        return _ret(
            claim_decl, "PENDENTE_REVISAO", "DECLARACAO_PUBLICA",
            "declaracao_publica_requer_verificacao", "titulo_normalizado"
        )

    # 4. Pendente: interrogativo / soft-lead / chamada explicativa
    if _tem_padrao(titulo_lower, PADROES_PENDENTE):
        return _ret(
            titulo, "PENDENTE_REVISAO", "VERDADEIRO_CURADO",
            "padrao_nao_factual_detectado", "titulo"
        )

    # 5. Pendente: sem verbo factual (fonte não prioritária)
    tem_verbo = _tem_verbo_factual(titulo_lower)
    if not tem_verbo and not fonte_forte:
        return _ret(
            titulo, "PENDENTE_REVISAO", "VERDADEIRO_CURADO",
            "sem_verbo_factual_fonte_nao_prioritaria", "titulo"
        )

    # 5b. Pendente: fonte forte mas título muito curto ou sem verbo
    if not tem_verbo and fonte_forte and len(titulo.split()) < 4:
        return _ret(
            titulo, "PENDENTE_REVISAO", "VERDADEIRO_CURADO",
            "titulo_muito_curto_sem_verbo", "titulo"
        )

    # 6. Gerar claim: converter verbo + normalizar
    claim = _converter_verbo_passado(titulo)
    claim = normalizar_pontuacao(claim)

    if len(claim) < CLAIM_MIN_CHARS:
        return _ret(
            claim, "PENDENTE_REVISAO", "VERDADEIRO_CURADO",
            "claim_muito_curto", "titulo_normalizado"
        )

    # 7. Aprovado
    return _ret(
        claim, "APROVADO_AUTO", "VERDADEIRO_CURADO",
        "claim_factual_gerado", "titulo_normalizado"
    )


# ── Testes v3 ────────────────────────────────────────────────────────────────
testes = [
    # APROVADO_AUTO esperado
    ("TSE divulgou calendário eleitoral de 2026",                            "TSE"),
    ("Câmara aprovou projeto que altera regras da reforma tributária",  "CAMARA_NOTICIAS"),
    ("Senado rejeitou PEC que amplia emendas parlamentares",                      "SENADO_NOTICIAS"),
    ("Banco Central manteve a taxa Selic em 10,5% ao ano",                        "BANCO_CENTRAL"),
    ("IBGE registrou crescimento de 2,3% no PIB",                                 "IBGE"),
    ("Lula sancionou lei que cria Universidade Federal Indígena",            "AGENCIA_BRASIL"),
    # DECLARACAO_PUBLICA esperado (eram APROVADO_AUTO em v2 — BUG CORRIGIDO)
    ("Lula diz sonhar em reverter privatizações de empresas estratégicas", "AGENCIA_BRASIL"),
    ("Governo avalia aumento de contratação pelo MEI com o fim da 6x1",        "AGENCIA_BRASIL"),
    ("Ministro garante aprovação da reforma ainda neste semestre",                        "AGENCIA_BRASIL"),
    ("Especialista afirma que urna eletrônica é segura",                       "AGENCIA_BRASIL"),
    ('É factoide do clã Bolsonaro, diz Alckmin',                               "AGENCIA_BRASIL"),
    ('"Ninguém respeita lambe-botas", diz Lula',                                    "AGENCIA_BRASIL"),
    # DESCARTADO esperado (opinião)
    ("Análise: o impacto da reforma tributária nas PMEs",              "AGENCIA_BRASIL"),
    ("Opinião: por que o STF errou na decisão",                        "AGENCIA_BRASIL"),
    # DESCARTADO esperado (página)
    ("Hotsite 135 anos do STF",                                                   "STF"),
    # PENDENTE esperado (chamada explicativa)
    ("Entenda a decisão do STF sobre deepfake eleitoral",                   "STF"),
    ("Como funciona o novo sistema de biometria eleitoral?",                      "TSE"),
]

print("=== Testes gerar_claim v3 ===\n")
print(f"{'Título (60c)':<62} {'Status':<22} {'label_detalhe':<22}")
print("-" * 110)
for titulo, portal in testes:
    r = gerar_claim(titulo, portal=portal)
    t = titulo[:60]
    print(f"{t:<62} {r['status_curadoria']:<22} {r['label_detalhe']:<22}")

=== Testes gerar_claim v3 ===

Título (60c)                                                   Status                 label_detalhe         
--------------------------------------------------------------------------------------------------------------
TSE divulgou calendário eleitoral de 2026                      APROVADO_AUTO          VERDADEIRO_CURADO     
Câmara aprovou projeto que altera regras da reforma tributár   APROVADO_AUTO          VERDADEIRO_CURADO     
Senado rejeitou PEC que amplia emendas parlamentares           APROVADO_AUTO          VERDADEIRO_CURADO     
Banco Central manteve a taxa Selic em 10,5% ao ano             APROVADO_AUTO          VERDADEIRO_CURADO     
IBGE registrou crescimento de 2,3% no PIB                      APROVADO_AUTO          VERDADEIRO_CURADO     
Lula sancionou lei que cria Universidade Federal Indígena      APROVADO_AUTO          VERDADEIRO_CURADO     
Lula diz sonhar em reverter privatizações de empresas estrat   PENDENTE_REVISAO       DECLARACA

In [31]:
def derivar_tipo_claim(status_curadoria: str, label_detalhe: str, motivo_status: str) -> str:
    """
    Classifica o claim em tipo semântico para análise e filtragem.
    Valores: FATO_INSTITUCIONAL | DECLARACAO_PUBLICA | CHAMADA_EXPLICATIVA | PAGINA_ESTATICA | OUTRO_PENDENTE
    """
    if status_curadoria == "APROVADO_AUTO":
        return "FATO_INSTITUCIONAL"
    if label_detalhe == "DECLARACAO_PUBLICA":
        return "DECLARACAO_PUBLICA"
    if motivo_status == "padrao_nao_factual_detectado":
        return "CHAMADA_EXPLICATIVA"
    if motivo_status == "pagina_ou_servico_institucional":
        return "PAGINA_ESTATICA"
    return "OUTRO_PENDENTE"


print("derivar_tipo_claim definida.")

derivar_tipo_claim definida.


## Coleta — RSS

In [32]:
FONTES_COM_FALHA = []   # registros de fontes que falharam (global, usado no relatório)


def coletar_feed(fonte_info: dict, max_itens: int = MAX_ITENS_POR_FONTE) -> list:
    portal     = fonte_info["portal"]
    tema       = fonte_info["tema"]
    subtema    = fonte_info["subtema"]
    url_feed   = fonte_info["url_feed"]
    pref_campo = fonte_info.get("preferencia_campo", "title_only")
    orig_qual  = fonte_info["origem_qualidade"]

    print(f"\n→ {portal} | {tema}/{subtema}")
    print(f"  {url_feed}")

    try:
        feed = feedparser.parse(
            url_feed,
            request_headers={"User-Agent": USER_AGENT}
        )
    except Exception as exc:
        msg = str(exc)[:120]
        print(f"  ERRO de conexão: {msg}")
        FONTES_COM_FALHA.append({"portal": portal, "url": url_feed, "erro": msg})
        return []

    if feed.bozo:
        tipo_exc = type(feed.bozo_exception).__name__ if feed.bozo_exception else "?"
        print(f"  Aviso: feed com problema ({tipo_exc})")

    n_total  = len(feed.entries)
    entries  = feed.entries[:max_itens]
    print(f"  Entradas: {n_total} (processando: {len(entries)})")

    if n_total == 0:
        FONTES_COM_FALHA.append({"portal": portal, "url": url_feed, "erro": "feed vazio (0 entradas)"})
        return []

    registros = []
    for entry in entries:
        titulo = entry.get("title", "")
        link   = entry.get("link", "")
        data_p = entry.get("published", entry.get("updated", ""))

        if pref_campo == "summary":
            resumo = entry.get("summary", "")
        elif pref_campo == "subtitle":
            resumo = entry.get("subtitle", "")
        else:
            resumo = ""

        registros.append({
            "titulo":              titulo,
            "resumo":              resumo,
            "url_origem":          link,
            "data_publicacao_raw": data_p,
            "portal_origem":       portal,
            "tema":                tema,
            "subtema":             subtema,
            "url_feed":            url_feed,
            "origem_qualidade":    orig_qual,
            "metodo_coleta":       "rss_feedparser",
            "data_coleta":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

    return registros


# ── Coleta com fallback, limite por fonte e limite global ────────────────
print("Iniciando coleta RSS...")
print(f"Configuração: max={MAX_ITENS_POR_FONTE}/fonte | total={MAX_ITENS_TOTAL} | delay={DELAY_SEGUNDOS}s\n")

todos_rss = []
fontes_principais = [f for f in FONTES_RSS if not f.get("_fallback", False)]
fontes_fallback   = {f["portal"]: f for f in FONTES_RSS if f.get("_fallback", False)}
portais_com_dados = set()

for i, fonte in enumerate(fontes_principais):
    if len(todos_rss) >= MAX_ITENS_TOTAL:
        print(f"\n→ Limite global ({MAX_ITENS_TOTAL}) atingido. Encerrando coleta RSS.")
        break

    regs = coletar_feed(fonte, max_itens=MAX_ITENS_POR_FONTE)

    if regs:
        todos_rss.extend(regs)
        portais_com_dados.add(fonte["portal"])
    elif fonte["portal"] in fontes_fallback and fonte["portal"] not in portais_com_dados:
        print(f"  → Tentando URL alternativa para {fonte['portal']}...")
        regs_fb = coletar_feed(fontes_fallback[fonte["portal"]], max_itens=MAX_ITENS_POR_FONTE)
        if regs_fb:
            todos_rss.extend(regs_fb)
            portais_com_dados.add(fonte["portal"])

    restante = MAX_ITENS_TOTAL - len(todos_rss)
    if i < len(fontes_principais) - 1 and restante > 0:
        time.sleep(DELAY_SEGUNDOS)

df_rss = pd.DataFrame(todos_rss)
print(f"\n{'='*55}")
print(f"RSS total bruto: {len(df_rss)} registros de {len(portais_com_dados)} portais")
if not df_rss.empty:
    print(df_rss["portal_origem"].value_counts().to_string())
if FONTES_COM_FALHA:
    print(f"\nFontes com falha: {len(FONTES_COM_FALHA)}")
    for f in FONTES_COM_FALHA:
        print(f"  {f['portal']:<20} {f['erro'][:80]}")

Iniciando coleta RSS...
Configuração: max=200/fonte | total=1000 | delay=1.5s


→ AGENCIA_BRASIL | política/governo_federal
  https://agenciabrasil.ebc.com.br/rss/politica/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | economia/economia_nacional
  https://agenciabrasil.ebc.com.br/rss/economia/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | saúde_pública/saude_publica
  https://agenciabrasil.ebc.com.br/rss/saude/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | educação/educacao
  https://agenciabrasil.ebc.com.br/rss/educacao/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | direitos_humanos/direitos_e_justica
  https://agenciabrasil.ebc.com.br/rss/direitos-humanos/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | judiciário/justica
  https://agenciabrasil.ebc.com.br/rss/justica/feed.xml
  Entradas: 10 (processando: 10)

→ AGENCIA_BRASIL | geral/geral
  https://agenciabrasil.ebc.com.br/rss/geral/feed.xml
  Entradas: 10 (pr

## Coleta — Banco Central (API JSON)

O Banco Central publica notícias e comunicados via API pública JSON.
Fonte: `https://www.bcb.gov.br`

Temas cobertos: inflação, Selic, Pix, reservas internacionais, política monetária,
supervisão bancária, indicadores econômicos oficiais.

In [33]:
URL_BCB_NOTICIAS = "https://www.bcb.gov.br/api/servico/sitebcb/noticiasRecentes/noticias"
BCB_BASE_URL     = "https://www.bcb.gov.br"

df_bcb = pd.DataFrame()  # inicializar vazio como fallback

print(f"Consultando Banco Central...")
print(f"  {URL_BCB_NOTICIAS}")

try:
    time.sleep(DELAY_ENTRE_REQUESTS)

    resp_bcb = requests.get(
        URL_BCB_NOTICIAS,
        params={"quantidade": 50},
        headers={"User-Agent": USER_AGENT},
        timeout=REQUEST_TIMEOUT
    )
    resp_bcb.raise_for_status()

    dados_bcb = resp_bcb.json()

    # Normalizar: a API pode retornar lista direta ou objeto com chave
    if isinstance(dados_bcb, list):
        itens_bcb = dados_bcb
    elif isinstance(dados_bcb, dict):
        # Tentar chaves comuns
        for chave in ["noticias", "items", "data", "results", "content"]:
            if chave in dados_bcb:
                itens_bcb = dados_bcb[chave]
                break
        else:
            itens_bcb = []
    else:
        itens_bcb = []

    print(f"  Status: {resp_bcb.status_code} | Itens: {len(itens_bcb)}")

    registros_bcb = []
    for item in itens_bcb:
        titulo = item.get("titulo", item.get("title", item.get("nome", "")))
        resumo = item.get("resumo", item.get("descricao", item.get("summary", "")))
        url    = item.get("url",    item.get("link",    item.get("href", "")))
        data_p = item.get("dataHoraPublicacao",
                 item.get("dataPublicacao",
                 item.get("data", item.get("date", ""))))

        if not titulo:
            continue

        # Completar URL relativa
        if url and not url.startswith("http"):
            url = BCB_BASE_URL + ("" if url.startswith("/") else "/") + url

        registros_bcb.append({
            "titulo":              str(titulo),
            "resumo":              str(resumo) if resumo else "",
            "url_origem":          str(url) if url else "",
            "data_publicacao_raw": str(data_p) if data_p else "",
            "portal_origem":       "BANCO_CENTRAL",
            "tema":                "economia",
            "subtema":             "banco_central",
            "url_feed":            URL_BCB_NOTICIAS,
            "origem_qualidade":    "ROTULO_ASSUMIDO_ALTO",
            "metodo_coleta":       "api_json",
            "data_coleta":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

    df_bcb = pd.DataFrame(registros_bcb)
    print(f"  {len(df_bcb)} registros estruturados.")

    if len(df_bcb) > 0:
        print("\n  Exemplos de títulos BCB:")
        for t in df_bcb["titulo"].head(4):
            print(f"    • {t[:100]}")

except requests.exceptions.HTTPError as e:
    print(f"  HTTP Error: {e}")
    print("  Continuando sem dados do Banco Central.")
except requests.exceptions.ConnectionError as e:
    print(f"  Erro de conexão: {e}")
    print("  Continuando sem dados do Banco Central.")
except Exception as e:
    print(f"  Erro inesperado: {type(e).__name__}: {e}")
    print("  Continuando sem dados do Banco Central.")

print(f"\nBCB: {len(df_bcb)} registros coletados.")

Consultando Banco Central...
  https://www.bcb.gov.br/api/servico/sitebcb/noticiasRecentes/noticias
  HTTP Error: 400 Client Error: Bad Request for url: https://www.bcb.gov.br/api/servico/sitebcb/noticiasRecentes/noticias?quantidade=50
  Continuando sem dados do Banco Central.

BCB: 0 registros coletados.


## Coleta Histórica — Sitemap XML e HTML Paginado

RSS feeds retornam apenas as últimas 10–200 entradas de cada portal. Para ampliar
o volume histórico sem depender de scraping agressivo, esta seção implementa duas
estratégias complementares:

| Estratégia | Portais | Mecanismo |
|---|---|---|
| **Sitemap XML** | Agência Brasil, STF, CNJ | Google News Sitemap (`<news:title>`, `<news:publication_date>`) |
| **HTML paginado** | Senado Notícias, TSE, Câmara Notícias | Extração de links de listagens com paginação via parâmetro de URL |

Regras mantidas:
- Delay de `DELAY_SEGUNDOS` entre requisições — sem coleta agressiva.
- Sem scraping do artigo completo — apenas título e URL.
- Filtragem por `DATA_MIN_HISTORICO` nos sitemaps.
- Falhas registradas em `FONTES_COM_FALHA` e exibidas no relatório.
- Classificação v3 (gerar_claim) aplicada identicamente ao RSS.

In [34]:
# ─── Fontes de Sitemap XML ────────────────────────────────────────────────
FONTES_SITEMAP = [
    {
        "portal":           "AGENCIA_BRASIL",
        "tema":             "geral",
        "subtema":          "historico_sitemap",
        "url_sitemap":      "https://agenciabrasil.ebc.com.br/sitemap.xml",
        "url_filtro":       "",
        "data_min":         DATA_MIN_HISTORICO,
        "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    {
        "portal":           "STF",
        "tema":             "judiciario",
        "subtema":          "decisoes_stf_historico",
        "url_sitemap":      "https://noticias.stf.jus.br/sitemap.xml",
        "url_filtro":       "",
        "data_min":         DATA_MIN_HISTORICO,
        "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
    {
        "portal":           "CNJ",
        "tema":             "judiciario",
        "subtema":          "cnj_historico",
        "url_sitemap":      "https://www.cnj.jus.br/sitemap.xml",
        "url_filtro":       "",
        "data_min":         DATA_MIN_HISTORICO,
        "origem_qualidade": "ROTULO_ASSUMIDO_ALTO",
    },
]

# ─── Fontes de listagem HTML paginada ─────────────────────────────────────
FONTES_LISTAGEM = [
    {
        "portal":            "SENADO_NOTICIAS",
        "tema":              "legislativo",
        "subtema":           "senado_historico",
        "url_base":          "https://www12.senado.leg.br/noticias/ultimas",
        "base_site":         "https://www12.senado.leg.br",
        "filtro_href":       "/noticias/materias/",
        "parametro_pagina":  "?p=",
        "pagina_inicial":    1,
        "max_paginas":       MAX_PAGINAS_POR_FONTE,
        "origem_qualidade":  "ROTULO_ASSUMIDO_ALTO",
    },
    {
        "portal":            "TSE",
        "tema":              "eleitoral",
        "subtema":           "eleicoes_historico",
        "url_base":          "https://www.tse.jus.br/comunicacao/noticias",
        "base_site":         "https://www.tse.jus.br",
        "filtro_href":       "/comunicacao/noticias/",
        "parametro_pagina":  "?page=",
        "pagina_inicial":    1,
        "max_paginas":       MAX_PAGINAS_POR_FONTE,
        "origem_qualidade":  "ROTULO_ASSUMIDO_ALTO",
    },
    {
        "portal":            "CAMARA_NOTICIAS",
        "tema":              "legislativo",
        "subtema":           "camara_historico",
        "url_base":          "https://www.camara.leg.br/noticias",
        "base_site":         "https://www.camara.leg.br",
        "filtro_href":       "/noticias/",
        "parametro_pagina":  "?pagina=",
        "pagina_inicial":    1,
        "max_paginas":       MAX_PAGINAS_POR_FONTE,
        "origem_qualidade":  "ROTULO_ASSUMIDO_ALTO",
    },
]

print(f"Fontes sitemap configuradas   : {len(FONTES_SITEMAP)}")
print(f"Fontes HTML paginado          : {len(FONTES_LISTAGEM)}")
print(f"Páginas máximas por fonte     : {MAX_PAGINAS_POR_FONTE}")
print(f"Data mínima (filtro sitemap)  : {DATA_MIN_HISTORICO}")

Fontes sitemap configuradas   : 3
Fontes HTML paginado          : 3
Páginas máximas por fonte     : 10
Data mínima (filtro sitemap)  : 2025-01-01


In [35]:
def _strip_ns(xml_text: str) -> str:
    """Remove declarações de namespace para simplificar o parsing com ET."""
    return re.sub(r' xmlns(?::\w+)?="[^"]*"', '', xml_text)


def coletar_sitemap(config_fonte: dict) -> list:
    """
    Coleta URLs de notícias via sitemap XML (Google News Sitemap ou sitemap index).
    Retorna lista de dicts compatíveis com o schema df_rss.
    """
    portal     = config_fonte["portal"]
    data_min   = config_fonte.get("data_min", DATA_MIN_HISTORICO)
    url_filtro = config_fonte.get("url_filtro", "")
    registros  = []

    def _parse(url: str, profundidade: int = 0):
        if profundidade > 2:
            return
        try:
            time.sleep(DELAY_SEGUNDOS)
            resp = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT})
            resp.raise_for_status()
            resp.encoding = "utf-8"
        except Exception as exc:
            FONTES_COM_FALHA.append({"portal": portal, "url": url, "erro": str(exc)[:120]})
            return

        try:
            root = ET.fromstring(_strip_ns(resp.text))
        except ET.ParseError as exc:
            FONTES_COM_FALHA.append({"portal": portal, "url": url, "erro": f"ParseError: {exc}"})
            return

        tag_root = root.tag.split("}")[-1] if "}" in root.tag else root.tag

        if tag_root == "sitemapindex":
            # sitemap index — buscar sub-sitemaps recursivamente
            for sm in root:
                loc_el = sm.find("loc")
                if loc_el is not None and (loc_el.text or "").strip():
                    _parse(loc_el.text.strip(), profundidade + 1)
            return

        # urlset — iterar <url>
        for url_el in root:
            tag = url_el.tag.split("}")[-1] if "}" in url_el.tag else url_el.tag
            if tag != "url":
                continue

            loc = titulo = data_pub = ""

            for child in url_el:
                ctag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
                if ctag == "loc":
                    loc = (child.text or "").strip()
                elif ctag == "title":
                    titulo = (child.text or "").strip()
                elif ctag == "news":
                    # Google News namespace: <news:news>
                    for nc in child:
                        nctag = nc.tag.split("}")[-1] if "}" in nc.tag else nc.tag
                        if nctag == "title":
                            titulo = (nc.text or "").strip()
                        elif nctag == "publication_date":
                            data_pub = (nc.text or "").strip()

            if not loc or not titulo:
                continue

            # filtro por URL
            if url_filtro and url_filtro not in loc:
                continue

            # filtro por data
            if data_min and data_pub:
                try:
                    if data_pub[:10] < data_min:
                        continue
                except Exception:
                    pass

            registros.append({
                "titulo":              titulo,
                "resumo":              "",
                "url_origem":          loc,
                "data_publicacao_raw": data_pub,
                "portal_origem":       portal,
                "tema":                config_fonte.get("tema", "geral"),
                "subtema":             config_fonte.get("subtema", "historico_sitemap"),
                "url_feed":            config_fonte["url_sitemap"],
                "origem_qualidade":    config_fonte.get("origem_qualidade", "ROTULO_ASSUMIDO_ALTO"),
                "metodo_coleta":       "sitemap_xml",
                "data_coleta":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

    _parse(config_fonte["url_sitemap"])
    print(f"  [SITEMAP] {portal}: {len(registros)} URLs coletadas")
    return registros


print("Função coletar_sitemap() definida.")

Função coletar_sitemap() definida.


In [36]:
class ExtractorLinksNoticia(HTMLParser):
    """Extrai pares (titulo, href) de páginas de listagem de notícias."""

    def __init__(self, filtro_href: str, base_site: str):
        super().__init__(convert_charrefs=True)
        self.filtro_href   = filtro_href
        self.base_site     = base_site.rstrip("/")
        self.links: list   = []
        self._in_link      = False
        self._cur_href     = None
        self._cur_text: list = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            attrs_d = dict(attrs)
            href = (attrs_d.get("href") or "").strip()
            if self.filtro_href in href:
                self._in_link  = True
                self._cur_href = (
                    href if href.startswith("http")
                    else self.base_site + "/" + href.lstrip("/")
                )
                self._cur_text = []

    def handle_endtag(self, tag):
        if tag == "a" and self._in_link:
            titulo = " ".join("".join(self._cur_text).split()).strip()
            if titulo and self._cur_href:
                self.links.append({"titulo": titulo, "url": self._cur_href})
            self._in_link  = False
            self._cur_href = None
            self._cur_text = []

    def handle_data(self, data):
        if self._in_link:
            self._cur_text.append(data)


def coletar_listagem_html(config_fonte: dict) -> list:
    """
    Coleta notícias via paginação HTML.
    Para a paginação quando encontra 3 páginas sem links novos.
    """
    portal           = config_fonte["portal"]
    url_base         = config_fonte["url_base"]
    base_site        = config_fonte["base_site"]
    filtro_href      = config_fonte["filtro_href"]
    param_pagina     = config_fonte["parametro_pagina"]
    pag_ini          = config_fonte.get("pagina_inicial", 1)
    max_pags         = config_fonte.get("max_paginas", MAX_PAGINAS_POR_FONTE)

    todos_links = []
    urls_vistas = set()
    vazias_cons = 0

    print(f"\n  [HTML] {portal}: paginação em {url_base}")

    for pagina in range(pag_ini, pag_ini + max_pags):
        url_pag = url_base if pagina == pag_ini else url_base + param_pagina + str(pagina)

        time.sleep(DELAY_SEGUNDOS)
        try:
            resp = requests.get(url_pag, timeout=REQUEST_TIMEOUT, headers={"User-Agent": USER_AGENT})
            resp.raise_for_status()
        except Exception as exc:
            print(f"    Pág {pagina}: erro — {str(exc)[:80]}")
            FONTES_COM_FALHA.append({"portal": portal, "url": url_pag, "erro": str(exc)[:120]})
            vazias_cons += 1
            if vazias_cons >= 3:
                print(f"    3 falhas consecutivas — encerrando {portal}")
                break
            continue

        parser = ExtractorLinksNoticia(filtro_href, base_site)
        try:
            parser.feed(resp.text)
        except Exception:
            pass

        novos = [l for l in parser.links if l["url"] not in urls_vistas]

        if not novos:
            vazias_cons += 1
            print(f"    Pág {pagina}: nenhum link novo ({vazias_cons} vazia(s) consecutiva(s))")
            if vazias_cons >= 3:
                print(f"    Encerrando paginação de {portal}")
                break
            continue

        vazias_cons = 0
        for l in novos:
            urls_vistas.add(l["url"])
            todos_links.append({
                "titulo":              l["titulo"],
                "resumo":              "",
                "url_origem":          l["url"],
                "data_publicacao_raw": "",
                "portal_origem":       portal,
                "tema":                config_fonte.get("tema", "legislativo"),
                "subtema":             config_fonte.get("subtema", "historico_html"),
                "url_feed":            url_base,
                "origem_qualidade":    config_fonte.get("origem_qualidade", "ROTULO_ASSUMIDO_ALTO"),
                "metodo_coleta":       "html_listagem",
                "data_coleta":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        print(f"    Pág {pagina}: {len(novos)} links novos (total: {len(todos_links)})")

    print(f"  [HTML] {portal}: {len(todos_links)} URLs coletadas")
    return todos_links


print("Funções ExtractorLinksNoticia e coletar_listagem_html() definidas.")

Funções ExtractorLinksNoticia e coletar_listagem_html() definidas.


In [37]:
registros_hist = []

# ── Sitemap XML ──────────────────────────────────────────────────────────
print("=== Coleta histórica — Sitemap XML ===")
for conf in FONTES_SITEMAP:
    regs = coletar_sitemap(conf)
    registros_hist.extend(regs)
n_sitemap = len(registros_hist)
print(f"\nTotal sitemap: {n_sitemap}")

# ── HTML paginado ────────────────────────────────────────────────────────
print("\n=== Coleta histórica — HTML paginado ===")
n_antes_html = len(registros_hist)
for conf in FONTES_LISTAGEM:
    regs = coletar_listagem_html(conf)
    registros_hist.extend(regs)
n_html = len(registros_hist) - n_antes_html
print(f"\nTotal HTML listagem: {n_html}")

# ── Converter para DataFrame ─────────────────────────────────────────────
if registros_hist:
    df_historico = pd.DataFrame(registros_hist)
    print(f"\nTotal histórico: {len(df_historico)} registros")
    print(f"\nDistribuição por portal:")
    print(df_historico["portal_origem"].value_counts().to_string())
    print(f"\nDistribuição por método:")
    print(df_historico["metodo_coleta"].value_counts().to_string())
else:
    df_historico = pd.DataFrame()
    print("\nNenhum registro histórico coletado (fontes indisponíveis).")

=== Coleta histórica — Sitemap XML ===
  [SITEMAP] AGENCIA_BRASIL: 0 URLs coletadas
  [SITEMAP] STF: 0 URLs coletadas
  [SITEMAP] CNJ: 0 URLs coletadas

Total sitemap: 0

=== Coleta histórica — HTML paginado ===

  [HTML] SENADO_NOTICIAS: paginação em https://www12.senado.leg.br/noticias/ultimas
    Pág 1: 13 links novos (total: 13)
    Pág 2: nenhum link novo (1 vazia(s) consecutiva(s))
    Pág 3: nenhum link novo (2 vazia(s) consecutiva(s))
    Pág 4: nenhum link novo (3 vazia(s) consecutiva(s))
    Encerrando paginação de SENADO_NOTICIAS
  [HTML] SENADO_NOTICIAS: 13 URLs coletadas

  [HTML] TSE: paginação em https://www.tse.jus.br/comunicacao/noticias
    Pág 1: 24 links novos (total: 24)
    Pág 2: nenhum link novo (1 vazia(s) consecutiva(s))
    Pág 3: nenhum link novo (2 vazia(s) consecutiva(s))
    Pág 4: nenhum link novo (3 vazia(s) consecutiva(s))
    Encerrando paginação de TSE
  [HTML] TSE: 24 URLs coletadas

  [HTML] CAMARA_NOTICIAS: paginação em https://www.camara.leg.br/n

## Consolidação — RAW unificado

Combina todas as fontes (RSS + BCB API) em um único DataFrame bruto.

In [38]:
frames = []
if not df_rss.empty:
    frames.append(df_rss)
if not df_bcb.empty:
    frames.append(df_bcb)
if "df_historico" in dir() and not df_historico.empty:
    frames.append(df_historico)

if not frames:
    raise RuntimeError("Nenhuma fonte retornou dados. Verificar conectividade.")

df_raw = pd.concat(frames, ignore_index=True)

n_rss  = len(df_rss)  if not df_rss.empty  else 0
n_bcb  = len(df_bcb)  if not df_bcb.empty  else 0
n_hist = len(df_historico) if ("df_historico" in dir() and not df_historico.empty) else 0

print(f"Total bruto consolidado: {len(df_raw)} registros")
print(f"  RSS (feedparser)   : {n_rss}")
print(f"  BCB API (json)     : {n_bcb}")
print(f"  Histórico          : {n_hist}")
print()
print("Distribuição por método de coleta:")
print(df_raw["metodo_coleta"].value_counts().to_string())
print()
print("Distribuição por portal:")
print(df_raw["portal_origem"].value_counts().to_string())
print()
print("Distribuição por tema:")
print(df_raw["tema"].value_counts().to_string())
print()
print(f"Registros com URL preenchida: {(df_raw['url_origem'].fillna('') != '').sum()}")

Total bruto consolidado: 183 registros
  RSS (feedparser)   : 120
  BCB API (json)     : 0
  Histórico          : 63

Distribuição por método de coleta:
metodo_coleta
rss_feedparser    120
html_listagem      63

Distribuição por portal:
portal_origem
AGENCIA_BRASIL     80
TSE                39
SENADO_NOTICIAS    28
CAMARA_NOTICIAS    26
STF                10

Distribuição por tema:
tema
legislativo         54
eleitoral           39
judiciário          20
política            10
economia            10
saúde_pública       10
educação            10
direitos_humanos    10
geral               10
tecnologia          10

Registros com URL preenchida: 183


## Salvar arquivo RAW

In [39]:
NOME_RAW     = f"checkai_autoral_verdadeiros_raw_{TIMESTAMP}.csv"
CAMINHO_RAW  = PASTA_RAW / NOME_RAW

df_raw.to_csv(CAMINHO_RAW, index=False, encoding="utf-8-sig")

print(f"Raw salvo em: {CAMINHO_RAW}")
print(f"Total       : {len(df_raw)} registros")
print(f"Data/hora   : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Raw salvo em: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_checkai_autoral\raw\checkai_autoral_verdadeiros_raw_2026-05-30_21-04-39.csv
Total       : 183 registros
Data/hora   : 30/05/2026 21:11:31


## Exploração dos dados raw

In [40]:
print(f"Shape : {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")

print(f"\nRegistros com resumo: {(df_raw['resumo'].fillna('') != '').sum()}")

print(f"\nExemplos de títulos por portal (3 cada):")
for portal in df_raw["portal_origem"].unique():
    print(f"\n  [{portal}]")
    for t in df_raw[df_raw["portal_origem"] == portal]["titulo"].head(3):
        print(f"    • {str(t)[:110]}")

Shape : (183, 11)
Colunas: ['titulo', 'resumo', 'url_origem', 'data_publicacao_raw', 'portal_origem', 'tema', 'subtema', 'url_feed', 'origem_qualidade', 'metodo_coleta', 'data_coleta']

Registros com resumo: 19

Exemplos de títulos por portal (3 cada):

  [AGENCIA_BRASIL]
    • Lula visita primeiro hospital oncológico interestadual do país
    • Lula diz sonhar em reverter privatizações de empresas estratégicas
    • É factoide do clã Bolsonaro para desviar do caso Master, diz Alckmin

  [STF]
    • Uso do Berçário – Advogadas
    • Relatório – 8 de janeiro
    • STF restabelece adicional de periculosidade a guardas municipais de Santo André (SP)

  [SENADO_NOTICIAS]
    • Dívidas em recorde assombram as famílias brasileiras
    • Papel dos líderes comunitários é destacado em sessão do Senado
    • Escola deve estimular descanso e abrir espaço para neurodivergentes, prevê projeto

  [TSE]
    • Nunes Marques conhece iniciativas do TRE-PR voltadas à inclusão indígena e à modernização da

## Curadoria — normalização e geração de claims

In [41]:
df = df_raw.copy()

df["titulo_limpo"] = df["titulo"].apply(remover_html)
df["resumo_limpo"] = df["resumo"].apply(remover_html)
df["data_publicacao"] = df["data_publicacao_raw"].apply(padronizar_data)

print("Limpeza HTML e normalização de datas aplicadas.")
print(f"Registros com data preenchida: {(df['data_publicacao'] != '').sum()} / {len(df)}")

Limpeza HTML e normalização de datas aplicadas.
Registros com data preenchida: 120 / 183


In [42]:
# gerar_claim v3 retorna 5 colunas
resultados = df.apply(
    lambda row: pd.Series(gerar_claim(
        titulo=row["titulo_limpo"],
        resumo=row["resumo_limpo"],
        portal=row["portal_origem"],
    )),
    axis=1
)

df = pd.concat([df, resultados], axis=1)

# Derivar tipo_claim
df["tipo_claim"] = df.apply(
    lambda r: derivar_tipo_claim(r["status_curadoria"], r["label_detalhe"], r["motivo_status"]),
    axis=1
)

print("=== Geração de claims concluída (v3) ===\n")
print("Distribuição status_curadoria:")
print(df["status_curadoria"].value_counts())
print("\nDistribuição label_detalhe:")
print(df["label_detalhe"].value_counts())
print("\nDistribuição tipo_claim:")
print(df["tipo_claim"].value_counts())
print("\nCross-tab portal × status:")
print(df.groupby(["portal_origem", "status_curadoria"]).size().unstack(fill_value=0))

df_decl = df[df["label_detalhe"] == "DECLARACAO_PUBLICA"]
if len(df_decl) > 0:
    print(f"\nExemplos DECLARACAO_PUBLICA ({len(df_decl)} total):")
    for _, row in df_decl.head(6).iterrows():
        print(f"  [{row['portal_origem']}] {row['titulo_limpo'][:100]}")

=== Geração de claims concluída (v3) ===

Distribuição status_curadoria:
status_curadoria
APROVADO_AUTO       140
PENDENTE_REVISAO     37
DESCARTADO            6
Name: count, dtype: int64

Distribuição label_detalhe:
label_detalhe
VERDADEIRO_CURADO     151
DECLARACAO_PUBLICA     26
DESCARTADO              6
Name: count, dtype: int64

Distribuição tipo_claim:
tipo_claim
FATO_INSTITUCIONAL     140
DECLARACAO_PUBLICA      26
OUTRO_PENDENTE           7
CHAMADA_EXPLICATIVA      6
PAGINA_ESTATICA          4
Name: count, dtype: int64

Cross-tab portal × status:
status_curadoria  APROVADO_AUTO  DESCARTADO  PENDENTE_REVISAO
portal_origem                                                
AGENCIA_BRASIL               66           0                14
CAMARA_NOTICIAS              18           2                 6
SENADO_NOTICIAS              20           0                 8
STF                           5           4                 1
TSE                          31           0                 8

Exem

## Controle de tamanho

In [43]:
df["texto_principal"]      = df["titulo_limpo"]
df["tamanho_chars"]        = df["texto_principal"].str.len().fillna(0).astype(int)
df["tamanho_chars_modelo"] = df["texto_principal_modelo"].fillna("").str.len().astype(int)
df["faixa_tamanho_modelo"] = df["tamanho_chars_modelo"].apply(calcular_faixa_tamanho)

df_aprov = df[df["status_curadoria"] == "APROVADO_AUTO"]
if len(df_aprov) > 0:
    lens = df_aprov["tamanho_chars_modelo"]
    print(f"APROVADO_AUTO — tamanho_chars_modelo:")
    print(f"  média={lens.mean():.0f}  mediana={lens.median():.0f}  mín={lens.min()}  máx={lens.max()}")

print(f"\nDistribuição faixa_tamanho_modelo:")
print(df["faixa_tamanho_modelo"].value_counts())

# Rebaixar APROVADO muito curto
mask_curto = (
    (df["tamanho_chars_modelo"] < CLAIM_MIN_CHARS)
    & (df["status_curadoria"] == "APROVADO_AUTO")
)
df.loc[mask_curto, "status_curadoria"] = "PENDENTE_REVISAO"
df.loc[mask_curto, "motivo_status"]    = "claim_muito_curto_apos_transformacao"
print(f"\nRebaixados para PENDENTE por tamanho: {mask_curto.sum()}")

APROVADO_AUTO — tamanho_chars_modelo:
  média=72  mediana=69  mín=41  máx=111

Distribuição faixa_tamanho_modelo:
faixa_tamanho_modelo
curto    183
Name: count, dtype: int64

Rebaixados para PENDENTE por tamanho: 0


## Filtros de qualidade

In [44]:
print("=== Filtros de qualidade ===\n")

# Filtro 1: claim vazio → DESCARTADO
mask_vazio = df["texto_principal_modelo"].fillna("").str.strip() == ""
df.loc[mask_vazio, "status_curadoria"] = "DESCARTADO"
df.loc[mask_vazio, "label_detalhe"]    = "DESCARTADO"
df.loc[mask_vazio, "motivo_status"]    = "claim_vazio"
print(f"Claims vazios → DESCARTADO          : {mask_vazio.sum()}")

# Filtro 2: sem URL → DESCARTADO
mask_sem_url = df["url_origem"].fillna("").str.strip() == ""
df.loc[mask_sem_url, "status_curadoria"] = "DESCARTADO"
df.loc[mask_sem_url, "label_detalhe"]    = "DESCARTADO"
df.loc[mask_sem_url, "motivo_status"]    = "sem_url_origem"
print(f"Sem URL → DESCARTADO                 : {mask_sem_url.sum()}")

# Filtro 3: sem data → anotar (não descartar)
mask_sem_data = df["data_publicacao"].fillna("").str.strip() == ""
print(f"Sem data (mantidos)                  : {mask_sem_data.sum()}")

n_bruto_antes_dedup = len(df)

# Filtro 4: dedup por claim normalizado
df["_chave_claim"] = (
    df["texto_principal_modelo"]
    .fillna("").str.lower().str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"[^\w\s]", "", regex=True)
)
antes = len(df)
df = df.drop_duplicates(subset=["_chave_claim"], keep="first").copy()
n_dedup_claim = antes - len(df)
print(f"Duplicatas por claim removidas       : {n_dedup_claim}")

# Filtro 5: dedup por URL
antes = len(df)
mask_url_ok = df["url_origem"].fillna("").str.strip() != ""
df_com_url  = df[mask_url_ok].drop_duplicates(subset=["url_origem"], keep="first")
df_sem_url  = df[~mask_url_ok]
df = pd.concat([df_com_url, df_sem_url], ignore_index=True)
n_dedup_url = antes - len(df)
print(f"Duplicatas por URL removidas         : {n_dedup_url}")

# Filtro 6: dedup por título normalizado
antes = len(df)
df["_chave_titulo"] = (
    df["texto_principal"]
    .fillna("").str.lower().str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"[^\w\s]", "", regex=True)
)
mask_url_ok2 = df["url_origem"].fillna("").str.strip() != ""
df_com_url2  = df[mask_url_ok2].drop_duplicates(subset=["_chave_titulo"], keep="first")
df_sem_url2  = df[~mask_url_ok2]
df = pd.concat([df_com_url2, df_sem_url2], ignore_index=True)
n_dedup_titulo = antes - len(df)
print(f"Duplicatas por título removidas      : {n_dedup_titulo}")

df = df.drop(columns=["_chave_claim", "_chave_titulo"])

n_total_dedup = n_dedup_claim + n_dedup_url + n_dedup_titulo
print(f"\nBrutos antes dedup                   : {n_bruto_antes_dedup}")
print(f"Total removido por dedup             : {n_total_dedup}")
print(f"Registros finais                     : {len(df)}")
print(f"\nDistribuição final status_curadoria:")
print(df["status_curadoria"].value_counts())
print(f"\nDistribuição label_detalhe:")
print(df["label_detalhe"].value_counts())

=== Filtros de qualidade ===

Claims vazios → DESCARTADO          : 2
Sem URL → DESCARTADO                 : 0
Sem data (mantidos)                  : 63
Duplicatas por claim removidas       : 33
Duplicatas por URL removidas         : 1
Duplicatas por título removidas      : 0

Brutos antes dedup                   : 183
Total removido por dedup             : 34
Registros finais                     : 149

Distribuição final status_curadoria:
status_curadoria
APROVADO_AUTO       114
PENDENTE_REVISAO     30
DESCARTADO            5
Name: count, dtype: int64

Distribuição label_detalhe:
label_detalhe
VERDADEIRO_CURADO     122
DECLARACAO_PUBLICA     22
DESCARTADO              5
Name: count, dtype: int64


## Matching de Queries por Tema/Subtema

Para cada registro, verifica qual `query` de `QUERIES_SUGERIDAS` aparece no título.
Gera três colunas novas: `query_matched`, `tema_query`, `subtema_query`.
Múltiplos matches → o primeiro da lista (mais específico) prevalece.

In [45]:
def _match_query(titulo: str) -> dict:
    titulo_lower = str(titulo).lower()
    for q in QUERIES_SUGERIDAS:
        if q["query"].lower() in titulo_lower:
            return {
                "query_matched": q["query"],
                "tema_query":    q["tema"],
                "subtema_query": q["subtema"],
            }
    return {"query_matched": "", "tema_query": "", "subtema_query": ""}

_match_res = df["texto_principal"].apply(lambda t: pd.Series(_match_query(t)))
df["query_matched"]  = _match_res["query_matched"]
df["tema_query"]     = _match_res["tema_query"]
df["subtema_query"]  = _match_res["subtema_query"]

n_com_query = (df["query_matched"] != "").sum()
n_sem_query = (df["query_matched"] == "").sum()
print(f"Registros com query matched  : {n_com_query} ({100*n_com_query/max(len(df),1):.1f}%)")
print(f"Registros sem query matched  : {n_sem_query}")

print(f"\nTop queries com mais registros:")
vc_q = df[df["query_matched"] != ""]["query_matched"].value_counts()
print(vc_q.head(20).to_string())

all_queries_set = {q["query"] for q in QUERIES_SUGERIDAS}
matched_set     = set(df[df["query_matched"] != ""]["query_matched"].unique())
sem_result      = sorted(all_queries_set - matched_set)
print(f"\nQueries sem nenhum resultado ({len(sem_result)}):")
if sem_result:
    for q in sem_result:
        print(f"  - {q}")
else:
    print("  (todas as queries tiveram pelo menos 1 resultado)")


Registros com query matched  : 60 (40.3%)
Registros sem query matched  : 89

Top queries com mais registros:
query_matched
TSE                    10
STF                    10
PL                      5
MEI                     4
SUS                     4
Lula                    3
INSS                    3
escala 6x1              3
Fies                    2
Novo                    2
PEC                     2
jornada de trabalho     2
eleições 2026           2
urna eletrônica         2
FGTS                    1
Enem                    1
Bolsonaro               1
8 de janeiro            1
pejotização             1
PT                      1

Queries sem nenhum resultado (72):
  - Alexandre de Moraes
  - André Janones
  - Arthur Lira
  - Auxílio Brasil
  - Banco Central
  - Bolsa Família
  - CLT
  - CPI
  - Centrão
  - Congresso Nacional
  - Covid
  - Câmara dos Deputados
  - Eduardo Bolsonaro
  - Foro de São Paulo
  - Guilherme Boulos
  - Jair Bolsonaro
  - MBL
  - MDB
  - Michelle Bolsonaro

## Montagem do schema curated

In [46]:
df["id_registro"]        = [str(__import__("uuid").uuid4()) for _ in range(len(df))]
df["label"]              = 1
df["pipeline_origem"]    = "checkai_autoral"
df["dataset_origem"]     = "CHECKAI_AUTORAL"
df["fonte_dataset"]      = "CHECKAI_AUTORAL"
df["referencia_dataset"] = df["url_origem"]
df["data_curadoria"]     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df["arquivo_raw_origem"] = NOME_RAW

COLUNAS_CURATED = [
    "id_registro",
    "texto_principal",
    "texto_principal_modelo",
    "label",
    "label_detalhe",
    "tipo_claim",
    "pipeline_origem",
    "dataset_origem",
    "portal_origem",
    "origem_texto",
    "origem_qualidade",
    "fonte_dataset",
    "referencia_dataset",
    "url_origem",
    "data_publicacao",
    "tema",
    "subtema",
    "tamanho_chars",
    "tamanho_chars_modelo",
    "faixa_tamanho_modelo",
    "status_curadoria",
    "motivo_status",
    "metodo_coleta",
    "query_matched",
    "tema_query",
    "subtema_query",
    "data_curadoria",
    "arquivo_raw_origem",
]

for col in COLUNAS_CURATED:
    if col not in df.columns:
        df[col] = ""

df_curated = df[COLUNAS_CURATED].reset_index(drop=True)

print(f"Schema curated: {df_curated.shape}")
print(f"Colunas ({len(df_curated.columns)}): {list(df_curated.columns)}")
assert (df_curated["label"] == 1).all(), "Erro: registros com label != 1"
print("\nValidação de integridade OK.")

Schema curated: (149, 28)
Colunas (28): ['id_registro', 'texto_principal', 'texto_principal_modelo', 'label', 'label_detalhe', 'tipo_claim', 'pipeline_origem', 'dataset_origem', 'portal_origem', 'origem_texto', 'origem_qualidade', 'fonte_dataset', 'referencia_dataset', 'url_origem', 'data_publicacao', 'tema', 'subtema', 'tamanho_chars', 'tamanho_chars_modelo', 'faixa_tamanho_modelo', 'status_curadoria', 'motivo_status', 'metodo_coleta', 'query_matched', 'tema_query', 'subtema_query', 'data_curadoria', 'arquivo_raw_origem']

Validação de integridade OK.


## Salvar arquivo curated

In [47]:
NOME_CURATED    = f"checkai_autoral_verdadeiros_curated_{TIMESTAMP}.csv"
CAMINHO_CURATED = PASTA_CURATED / NOME_CURATED

df_curated.to_csv(CAMINHO_CURATED, index=False, encoding="utf-8-sig")

n_aprov   = (df_curated["status_curadoria"] == "APROVADO_AUTO").sum()
n_pend    = (df_curated["status_curadoria"] == "PENDENTE_REVISAO").sum()
n_desc    = (df_curated["status_curadoria"] == "DESCARTADO").sum()
n_decl    = (df_curated["label_detalhe"]    == "DECLARACAO_PUBLICA").sum()
n_fato    = (df_curated["tipo_claim"]        == "FATO_INSTITUCIONAL").sum()
n_chamada = (df_curated["tipo_claim"]        == "CHAMADA_EXPLICATIVA").sum()
n_pagina  = (df_curated["tipo_claim"]        == "PAGINA_ESTATICA").sum()

print(f"Curated salvo : {CAMINHO_CURATED}")
print()
print(f"Total                    : {len(df_curated)}")
print(f"  APROVADO_AUTO          : {n_aprov}  (tipo: FATO_INSTITUCIONAL={n_fato})")
print(f"  PENDENTE_REVISAO       : {n_pend}")
print(f"    └ DECLARACAO_PUBLICA  : {n_decl}")
print(f"    └ CHAMADA_EXPLICATIVA : {n_chamada}")
print(f"  DESCARTADO             : {n_desc}  (páginas: {n_pagina})")

Curated salvo : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_checkai_autoral\curated\checkai_autoral_verdadeiros_curated_2026-05-30_21-04-39.csv

Total                    : 149
  APROVADO_AUTO          : 114  (tipo: FATO_INSTITUCIONAL=114)
  PENDENTE_REVISAO       : 30
    └ DECLARACAO_PUBLICA  : 22
    └ CHAMADA_EXPLICATIVA : 5
  DESCARTADO             : 5  (páginas: 4)


## Verificação de qualidade

In [48]:
print("=== Verificação de qualidade (v3) ===\n")

COLS_OBRIG = ["id_registro", "texto_principal", "texto_principal_modelo",
              "label", "label_detalhe", "tipo_claim", "pipeline_origem",
              "url_origem", "status_curadoria", "origem_qualidade"]
print("Nulos por coluna obrigatória:")
print(df_curated[COLS_OBRIG].isnull().sum())

df_aprov = df_curated[df_curated["status_curadoria"] == "APROVADO_AUTO"]
if len(df_aprov) > 0:
    lens = df_aprov["tamanho_chars_modelo"]
    print(f"\nEstatísticas APROVADO_AUTO (tamanho_chars_modelo):")
    print(f"  média={lens.mean():.0f}  mediana={lens.median():.0f}  mín={lens.min()}  máx={lens.max()}")
    print(f"\nPor portal (APROVADO_AUTO):")
    print(df_aprov["portal_origem"].value_counts())

print(f"\n--- APROVADO_AUTO / FATO_INSTITUCIONAL ({len(df_aprov)}) ---")
for _, row in df_aprov.head(8).iterrows():
    print(f"  [{row['portal_origem']:<18}] {row['texto_principal_modelo'][:100]}")

df_decl = df_curated[df_curated["label_detalhe"] == "DECLARACAO_PUBLICA"]
print(f"\n--- DECLARACAO_PUBLICA ({len(df_decl)}) ---")
for _, row in df_decl.head(6).iterrows():
    print(f"  [{row['portal_origem']:<18}] {row['texto_principal_modelo'][:100]}")

df_cham = df_curated[df_curated["tipo_claim"] == "CHAMADA_EXPLICATIVA"]
print(f"\n--- CHAMADA_EXPLICATIVA ({len(df_cham)}) ---")
for _, row in df_cham.head(5).iterrows():
    print(f"  [{row['portal_origem']:<18}] {row['texto_principal'][:90]}")

df_pag = df_curated[df_curated["tipo_claim"] == "PAGINA_ESTATICA"]
print(f"\n--- PAGINA_ESTATICA ({len(df_pag)}) ---")
for _, row in df_pag.head(5).iterrows():
    print(f"  [{row['portal_origem']:<18}] {row['texto_principal'][:90]}")

df_outro = df_curated[df_curated["tipo_claim"] == "OUTRO_PENDENTE"]
print(f"\n--- OUTRO_PENDENTE ({len(df_outro)}) ---")
for _, row in df_outro.head(5).iterrows():
    print(f"  [{row['portal_origem']:<18}] {row['texto_principal'][:80]} [{row['motivo_status']}]")

df_desc = df_curated[df_curated["status_curadoria"] == "DESCARTADO"]
print(f"\n--- DESCARTADO ({len(df_desc)}) ---")
for _, row in df_desc.head(6).iterrows():
    print(f"  [{row['portal_origem']:<18}] {str(row['texto_principal'])[:80]} [{row['motivo_status']}]")

=== Verificação de qualidade (v3) ===

Nulos por coluna obrigatória:
id_registro               0
texto_principal           0
texto_principal_modelo    0
label                     0
label_detalhe             0
tipo_claim                0
pipeline_origem           0
url_origem                0
status_curadoria          0
origem_qualidade          0
dtype: int64

Estatísticas APROVADO_AUTO (tamanho_chars_modelo):
  média=71  mediana=69  mín=41  máx=111

Por portal (APROVADO_AUTO):
portal_origem
AGENCIA_BRASIL     56
TSE                21
CAMARA_NOTICIAS    18
SENADO_NOTICIAS    14
STF                 5
Name: count, dtype: int64

--- APROVADO_AUTO / FATO_INSTITUCIONAL (114) ---
  [AGENCIA_BRASIL    ] Lula visita primeiro hospital oncológico interestadual do país.
  [AGENCIA_BRASIL    ] Castro desiste de candidatura ao Senado após ser alvo de ações da PF.
  [AGENCIA_BRASIL    ] Lula sanciona lei que criou Universidade Federal Indígena.
  [AGENCIA_BRASIL    ] Governo prorrogou descontos no q

## Relatório final

In [49]:
import glob as _glob, os as _os

# ── Referência v2 para comparação ──────────────────────────────────────────
arquivos_curated = sorted(_glob.glob(str(PASTA_CURATED / "checkai_autoral_verdadeiros_curated_*.csv")))
df_v2_ref    = None
nome_v2_ref  = "N/A"
if len(arquivos_curated) >= 2:
    arq_v2 = arquivos_curated[-2]
    try:
        df_v2_ref   = pd.read_csv(arq_v2, encoding="utf-8-sig")
        nome_v2_ref = _os.path.basename(arq_v2)
        print(f"Referência anterior: {nome_v2_ref} ({len(df_v2_ref)} registros)")
    except Exception as e:
        print(f"Não foi possível carregar referência: {e}")
elif len(arquivos_curated) == 1:
    print("Apenas um curated encontrado (atual). Sem referência anterior.")

# ── Sub-dataframes ─────────────────────────────────────────────────────────
df_aprov  = df_curated[df_curated["status_curadoria"] == "APROVADO_AUTO"]
df_pend   = df_curated[df_curated["status_curadoria"] == "PENDENTE_REVISAO"]
df_desc   = df_curated[df_curated["status_curadoria"] == "DESCARTADO"]
df_decl   = df_curated[df_curated["label_detalhe"]    == "DECLARACAO_PUBLICA"]
df_cham   = df_curated[df_curated["tipo_claim"]        == "CHAMADA_EXPLICATIVA"]
df_pag    = df_curated[df_curated["tipo_claim"]        == "PAGINA_ESTATICA"]
df_outro  = df_curated[df_curated["tipo_claim"]        == "OUTRO_PENDENTE"]
df_com_q  = df_curated[df_curated["query_matched"]      != ""]
df_sem_q  = df_curated[df_curated["query_matched"]      == ""]
lens      = df_aprov["tamanho_chars_modelo"] if len(df_aprov) > 0 else pd.Series([0])
_falhas   = FONTES_COM_FALHA if "FONTES_COM_FALHA" in dir() else []

# Variável de dedup (pode não existir se filtros não rodaram)
_n_bruto_antes_dedup = n_bruto_antes_dedup if "n_bruto_antes_dedup" in dir() else "N/A"

# ── Helpers ────────────────────────────────────────────────────────────────
def fmt_ex(sub, n=5, col="texto_principal_modelo", col2=None):
    linhas = []
    for _, r in sub.head(n).iterrows():
        linha = f"- [{r['portal_origem']}] {str(r[col])[:120]}"
        if col2:
            linha += f"  *({r[col2]})*"
        linhas.append(linha)
    return "\n".join(linhas) if linhas else "*Nenhum.*"

def fmt_ex_por_portal(sub, n_por_portal=2, col="texto_principal_modelo"):
    linhas = []
    for portal in sub["portal_origem"].unique():
        linhas.append(f"\n**{portal}**")
        for _, r in sub[sub["portal_origem"] == portal].head(n_por_portal).iterrows():
            linhas.append(f"- {str(r[col])[:120]}")
    return "\n".join(linhas) if linhas else "*Nenhum.*"

# ── Seção comparação v2 ────────────────────────────────────────────────────
if df_v2_ref is not None:
    v2_aprov = (df_v2_ref["status_curadoria"] == "APROVADO_AUTO").sum()
    v2_pend  = (df_v2_ref["status_curadoria"] == "PENDENTE_REVISAO").sum()
    v2_desc  = (df_v2_ref["status_curadoria"] == "DESCARTADO").sum()
    v2_decl  = (df_v2_ref.get("label_detalhe", pd.Series()) == "DECLARACAO_PUBLICA").sum()
    v2_total = len(df_v2_ref)
    sec_comp = f"""
## Comparação com execução anterior

| Métrica | Execução anterior | Esta execução | Δ |
|---|---|---|---|
| Total curated | {v2_total} | {len(df_curated)} | {len(df_curated) - v2_total:+d} |
| APROVADO_AUTO | {v2_aprov} | {len(df_aprov)} | {len(df_aprov) - v2_aprov:+d} |
| PENDENTE_REVISAO | {v2_pend} | {len(df_pend)} | {len(df_pend) - v2_pend:+d} |
| DECLARACAO_PUBLICA | {v2_decl} | {len(df_decl)} | — |
| DESCARTADO | {v2_desc} | {len(df_desc)} | {len(df_desc) - v2_desc:+d} |

*Referência: `{nome_v2_ref}`*
"""
else:
    sec_comp = "\n*Primeira execução ou referência anterior não encontrada.*\n"


# ── Cobertura por Query ──────────────────────────────────────────────────
_vc_query      = df_curated[df_curated["query_matched"] != ""]["query_matched"].value_counts()
_all_q         = {q["query"] for q in QUERIES_SUGERIDAS}
_matched_q     = set(df_curated[df_curated["query_matched"] != ""]["query_matched"].unique())
_sem_result_q  = sorted(_all_q - _matched_q)
_sem_result_str = "\n".join(f"- {q}" for q in _sem_result_q) if _sem_result_q else "*Todas as queries tiveram resultado.*"

sec_queries = f"""
## Cobertura por Query

| Métrica | Valor |
|---|---|
| Registros com query matched | {len(df_com_q)} ({100*len(df_com_q)/max(len(df_curated),1):.1f}%) |
| Registros sem query matched | {len(df_sem_q)} |
| Queries com resultado | {len(_matched_q)} / {len(_all_q)} |
| Queries sem resultado | {len(_sem_result_q)} |

### Top queries — mais registros matched

```
{_vc_query.head(30).to_string() if len(_vc_query) > 0 else "Nenhum."}
```

### Cobertura por tema_query

```
{df_curated[df_curated["tema_query"] != ""]["tema_query"].value_counts().to_string() if len(df_com_q) > 0 else "Nenhum."}
```

### Queries sem nenhum resultado ({len(_sem_result_q)})

{_sem_result_str}
"""

# ── Seção fontes com falha ─────────────────────────────────────────────────
if _falhas:
    linhas_falha = ["\n".join([f"- **{f['portal']}**: `{f['url']}`  \n  Erro: {f['erro']}" for f in _falhas])]
    sec_falhas = "\n## Alertas de Fontes com Falha\n\n" + linhas_falha[0]
else:
    sec_falhas = "\n## Alertas de Fontes com Falha\n\n*Nenhuma falha registrada.*"

# ── Top portais APROVADO_AUTO ─────────────────────────────────────────────
top_portais = df_aprov["portal_origem"].value_counts() if len(df_aprov) > 0 else pd.Series()
top_portais_str = top_portais.to_string() if len(top_portais) > 0 else "Nenhum."

# ── Meta próxima execução ─────────────────────────────────────────────────
meta_aprov_atual = len(df_aprov)
meta_prox = max(200, meta_aprov_atual + 50) if meta_aprov_atual < 500 else "Meta atingida (≥500)"

# ── Gerar relatório ────────────────────────────────────────────────────────
relatorio = f"""# Relatório — Pipeline CheckAI Autoral (Verdadeiros) v3

**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
**Timestamp:** {TIMESTAMP}
**Raw:** `{NOME_RAW}`
**Curated:** `{NOME_CURATED}`
**Parâmetros:** max_por_fonte={MAX_ITENS_POR_FONTE} | max_total={MAX_ITENS_TOTAL} | delay={DELAY_SEGUNDOS}s

---

## Resumo

| Métrica | Valor |
|---|---|
| Total bruto coletado | {len(df_raw)} |
| └ RSS (feedparser) | {n_rss} |
| └ BCB API (json) | {n_bcb} |
| └ Histórico (sitemap + HTML) | {n_hist} |
| Total após deduplicação | {_n_bruto_antes_dedup} → {len(df_curated)} |
| **APROVADO_AUTO (FATO_INSTITUCIONAL)** | **{len(df_aprov)}** |
| PENDENTE_REVISAO (total) | {len(df_pend)} |
| └ DECLARACAO_PUBLICA | {len(df_decl)} |
| └ CHAMADA_EXPLICATIVA | {len(df_cham)} |
| └ OUTRO_PENDENTE | {len(df_outro)} |
| DESCARTADO (total) | {len(df_desc)} |
| └ PAGINA_ESTATICA | {len(df_pag)} |
| **Total no curated** | **{len(df_curated)}** |

---

## Por método de coleta

```
{df_raw["metodo_coleta"].value_counts().to_string()}
```

## Por portal_origem (bruto)

```
{df_curated['portal_origem'].value_counts().to_string()}
```

## Top portais — APROVADO_AUTO

```
{top_portais_str}
```

## Por tema

```
{df_curated['tema'].value_counts().to_string()}
```

## Por subtema

```
{df_curated['subtema'].value_counts().to_string()}
```

## Por status_curadoria

```
{df_curated['status_curadoria'].value_counts().to_string()}
```

## Por label_detalhe

```
{df_curated['label_detalhe'].value_counts().to_string()}
```

## Por tipo_claim

```
{df_curated['tipo_claim'].value_counts().to_string()}
```

## Por motivo_status

```
{df_curated['motivo_status'].value_counts().to_string()}
```

---
{sec_comp}
---
{sec_queries}

## Estatísticas de tamanho (APROVADO_AUTO)

| Métrica | Valor |
|---|---|
| Média (chars) | {lens.mean():.0f} |
| Mediana (chars) | {lens.median():.0f} |
| Mínimo | {lens.min()} |
| Máximo | {lens.max()} |

---

## Exemplos APROVADO_AUTO por portal

{fmt_ex_por_portal(df_aprov, n_por_portal=3)}

## Exemplos DECLARACAO_PUBLICA

{fmt_ex(df_decl, n=6, col2='motivo_status')}

## Exemplos CHAMADA_EXPLICATIVA

{fmt_ex(df_cham, n=5, col='texto_principal', col2='motivo_status')}

## Exemplos PAGINA_ESTATICA (DESCARTADO)

{fmt_ex(df_pag, n=5, col='texto_principal', col2='motivo_status')}

## Exemplos OUTRO_PENDENTE

{fmt_ex(df_outro, n=5, col='texto_principal', col2='motivo_status')}

---
{sec_falhas}

---

## Limitações metodológicas

1. **Baseado em título de RSS:** não faz scraping do artigo completo.
2. **Claim generation por regras determinísticas:** sem IA. Verbos não mapeados → PENDENTE.
3. **DECLARACAO_PUBLICA não verificadas:** a fala é real, mas o conteúdo pode ser impreciso.
4. **Buffer RSS:** cada feed retorna últimas 10–200 entradas. Sem cobertura histórica automática.
5. **BCB API:** endpoint pode mudar sem aviso. Falha é silenciosa.
6. **Gov.br feeds:** alguns ministérios podem não expor RSS — falha tratada graciosamente.
7. **ROTULO_ASSUMIDO_ALTO:** baseado na confiabilidade da fonte, não verificação por claim.

---

## Critérios para futura montagem da V4

- Usar automaticamente apenas `status_curadoria = APROVADO_AUTO` (`tipo_claim = FATO_INSTITUCIONAL`).
- `DECLARACAO_PUBLICA` só entram após revisão manual e promoção para `APROVADO_MANUAL`.
- `PENDENTE_REVISAO` não entram no treino principal sem revisão.
- `DESCARTADO` nunca entram.
- Manter `CHECKAI_AUTORAL` como `dataset_origem` separado.
- Manter `origem_qualidade = ROTULO_ASSUMIDO_ALTO` para APROVADO_AUTO.
- Usar `ROTULO_FORTE_MANUAL` apenas se houver revisão humana explícita documentada.

---

## Recomendação para próxima execução

- APROVADO_AUTO nesta execução: **{meta_aprov_atual}**
- Meta recomendada para a próxima execução: **{meta_prox}**
- Executar novamente em 2–4 semanas para cobertura temporal.
- Revisar manualmente os **{len(df_decl)}** registros DECLARACAO_PUBLICA antes de incluir na V4.
- Investigar fontes com falha: {len(_falhas)} fonte(s) precisam de verificação de URL.

---

## Conclusão metodológica

A expansão mantém os critérios rígidos da versão v3 e busca aumentar o volume da base própria
sem comprometer a qualidade do label positivo. A coleta ampliada prioriza fatos institucionais
objetivos de fontes oficiais e jornalísticas confiáveis, mantendo rastreabilidade por URL e
separando declarações públicas, chamadas explicativas e páginas estáticas do conjunto
automaticamente utilizável na futura V4.

---
*Gerado por `src/coleta_checkai_autoral_verdadeiros.ipynb` v3 (expansão)*
"""

NOME_REL    = f"relatorio_checkai_autoral_verdadeiros_{TIMESTAMP}.md"
CAMINHO_REL = PASTA_FINAL / NOME_REL

with open(CAMINHO_REL, "w", encoding="utf-8") as f:
    f.write(relatorio)

print(f"Relatório: {CAMINHO_REL}")
print()
print("=" * 60)
print("RESUMO FINAL")
print("=" * 60)
print(f"Total coletado (bruto)       : {len(df_raw)}")
print(f"  RSS                        : {n_rss}")
print(f"  BCB API                    : {n_bcb}")
print(f"  Histórico                  : {n_hist}")
print(f"Após deduplicação            : {len(df_curated)}")
print(f"APROVADO_AUTO                : {len(df_aprov)}  ← FATO_INSTITUCIONAL")
print(f"PENDENTE_REVISAO             : {len(df_pend)}")
print(f"  DECLARACAO_PUBLICA         : {len(df_decl)}")
print(f"  CHAMADA_EXPLICATIVA        : {len(df_cham)}")
print(f"  OUTRO_PENDENTE             : {len(df_outro)}")
print(f"DESCARTADO                   : {len(df_desc)}")
print(f"  PAGINA_ESTATICA            : {len(df_pag)}")
print()
print("Fontes com mais APROVADO_AUTO:")
print(top_portais.head(5).to_string())
print()
print(f"Fontes com falha             : {len(_falhas)}")
if _falhas:
    for f in _falhas:
        print(f"  {f['portal']:<20} {f['erro'][:60]}")
print()
print("Recomendação para V4:")
print("  Usar somente APROVADO_AUTO (FATO_INSTITUCIONAL).")
print("  DECLARACAO_PUBLICA requer revisão manual antes de incluir.")

Referência anterior: checkai_autoral_verdadeiros_curated_2026-05-30_20-09-20.csv (149 registros)
Relatório: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_checkai_autoral\final\relatorio_checkai_autoral_verdadeiros_2026-05-30_21-04-39.md

RESUMO FINAL
Total coletado (bruto)       : 183
  RSS                        : 120
  BCB API                    : 0
  Histórico                  : 63
Após deduplicação            : 149
APROVADO_AUTO                : 114  ← FATO_INSTITUCIONAL
PENDENTE_REVISAO             : 30
  DECLARACAO_PUBLICA         : 22
  CHAMADA_EXPLICATIVA        : 5
  OUTRO_PENDENTE             : 4
DESCARTADO                   : 5
  PAGINA_ESTATICA            : 4

Fontes com mais APROVADO_AUTO:
portal_origem
AGENCIA_BRASIL     56
TSE                21
CAMARA_NOTICIAS    18
SENADO_NOTICIAS    14
STF                 5

Fontes com falha             : 76
  AGENCIA_BRASIL       feed vazio (0 entradas)
  CAMARA_NOTICIAS      feed vazio (0 entradas)
  CAMARA_NOTICIAS      feed vazi

---

## Conclusão Metodológica

> A versão v3 do pipeline autoral endurece os critérios de aprovação automática para evitar
> que declarações públicas, chamadas explicativas ou páginas institucionais sejam tratadas
> como fatos verdadeiros diretamente utilizáveis no treino. Essa separação reduz ruído no
> label positivo e fortalece a rastreabilidade metodológica da futura V4.

**Correções v3 vs v2:**
- Verbos como `diz`, `avalia`, `garante`, `afirma`, `critica` e outros 25 verbos declaratórios
  são agora verificados por **palavra exata** (`\b`) com prioridade absoluta sobre verbos factuais.
- Casos como _"Governo avalia aumento pelo MEI"_ e _"Lula diz sonhar em reverter privatizações"_
  que eram APROVADO_AUTO em v2 são corretamente classificados como DECLARACAO_PUBLICA em v3.
- `PADROES_PAGINA` descarta títulos de páginas estáticas (hotsite, relatório, museu).
- `texto_principal_modelo` de DECLARACAO_PUBLICA normaliza o verbo para passado (ex.: _diz → disse_).
- Nova coluna `tipo_claim` permite filtrar exatamente FATO_INSTITUCIONAL para a futura V4.